# Figures & tables for the thesis — generated from the results files
Every figure and table used in the Methodology (Ch. 3) and Results (Ch. 4) chapters is produced here, in pipeline order, from the result files on Drive. Ids follow **chapter → pipeline stage**: `F3.x` / `T3.x` (methodology), `F4.x` / `T4.x` (results). Files: `figures/fig_<id>_<stage>.{pdf,png}`, `tables/tab_<id>_<stage>.{csv,tex,png}`; the index of all items with draft captions is written to `figures/INDEX.md`.

Sections: **L0** author's annotation pass · **Stage 1** data · **Stage 2** extraction · **Stage 3** pattern mining · **Stage 4** feature selection · **Stage 5–6** ranking & evaluation protocol · **Results 4.1–4.7**.

## L0 — Author's independent annotation pass (Pakistan evaluation set)
Writes a blind annotation sheet (`row, sentence` only, shuffled) for the author to fill in; the ingest cell computes Cohen's κ against the collection-time labels and stores the author's labels in the evaluation file.

In [ ]:
# Setup — Drive mount, paths, chapter map. Every figure/table below is generated from a results file on Drive; nothing is typed in.
from google.colab import drive
drive.mount('/content/drive')
import os, json, re, time
import numpy as np, pandas as pd
from collections import Counter
ROOT = '/content/drive/MyDrive/geoproject'
FIG, TAB = f'{ROOT}/figures', f'{ROOT}/tables'
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
CH = {'method': 3, 'results': 4}          # thesis chapter numbers; change here if the template differs
INDEX = []                                 # (id, file, stage, caption, source) -> figures/INDEX.md
print('ready:', ROOT)

In [ ]:
# L0.1 — write the blind annotation sheet (sentence only, shuffled) + the tag list
import json, os, numpy as np, pandas as pd
out_x = f'{ROOT}/eval/pk_annotation_sheet.xlsx'
if os.path.exists(out_x):
    print(f'{out_x} already exists — not overwriting (it holds the filled annotation). Delete it first to regenerate.')
else:
    pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
    LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
    rng = np.random.default_rng(2026)
    sheet = pd.DataFrame({'row': [r['row'] for r in pk], 'sentence': [r['sentence'] for r in pk]}).iloc[rng.permutation(len(pk))].reset_index(drop=True)
    sheet['tag'] = ''; sheet['alt_tag'] = ''; sheet['unsure'] = ''
    sheet.to_csv(f'{ROOT}/eval/pk_annotation_sheet.csv', index=False)
    pd.DataFrame({'tag': sorted(LEX)}).to_csv(f'{ROOT}/eval/pk_annotation_taglist.csv', index=False)
    with pd.ExcelWriter(out_x) as xw:
        sheet.to_excel(xw, sheet_name='annotate', index=False); pd.DataFrame({'tag': sorted(LEX)}).to_excel(xw, sheet_name='tag_list', index=False)
    print(f'wrote eval/pk_annotation_sheet.csv + .xlsx ({len(sheet)} rows, shuffled) and eval/pk_annotation_taglist.csv ({len(LEX)} tags)')

In [ ]:
# L0.2 — ingest the author's filled sheet -> Cohen's kappa vs collection-time labels -> store as annotator_b_tag (run after the sheet is complete)
import json, os, numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
p_x, p_c = f'{ROOT}/eval/pk_annotation_sheet.xlsx', f'{ROOT}/eval/pk_annotation_sheet.csv'
if os.path.getmtime(p_x) > os.path.getmtime(p_c):
    book = pd.read_excel(p_x, sheet_name=None)                       # whichever sheet holds the row / sentence / tag columns
    sh = next(df for name, df in book.items() if {'row', 'tag'} <= set(map(str, df.columns))); print('read sheet:', next(n for n, df in book.items() if df is sh))
else: sh = pd.read_csv(p_c)
sh['tag'] = sh['tag'].astype(str).str.strip().str.lower().replace({'nan': ''})
LEX = set(json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon'])
missing = sh[sh.tag == '']; bad = sh[(sh.tag != '') & ~sh.tag.isin(LEX)]
assert len(missing) == 0, f'{len(missing)} rows still empty: {missing.row.tolist()[:20]}'
assert len(bad) == 0, f'tags outside the 227-tag list (check spelling): {bad[["row","tag"]].values.tolist()[:20]}'
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); B = dict(zip(sh.row.astype(int), sh.tag))
for r in pk: r['annotator_b_tag'] = B[r['row']]
a = [r.get('collection_tag', r['gold_tag']) for r in pk]; b = [r['annotator_b_tag'] for r in pk]   # kappa is always A (collection-time) vs B, even after adjudication
dist = Counter(a); n3 = {t for t, c in dist.items() if c >= 3}; sub = [(x, y) for x, y in zip(a, b) if x in n3]
agree = sum(x == y for x, y in zip(a, b)); alt_ok = sum(1 for r, x, y in zip(pk, a, b) if y != x and y in r['alt_tags']); unc = sum(1 for r, x, y in zip(pk, a, b) if y != x and r['uncertain'])
res = {'kappa_all': cohen_kappa_score(a, b), 'agreement_all': agree / len(pk), 'n': len(pk), 'kappa_n3': cohen_kappa_score(*zip(*sub)), 'n_n3': len(sub),
       'disagreements': [{'row': r['row'], 'A': x, 'B': y, 'alt': r['alt_tags'], 'uncertain': r['uncertain'], 'sentence': r['sentence']} for r, x, y in zip(pk, a, b) if x != y],
       'n_disagree_in_alt': alt_ok, 'n_disagree_uncertain': unc,
       'protocol': 'annotator A = collection-time labels; annotator B = the author, blind pass from the sentence text alone over the closed 227-tag list'}
print(f"Cohen's kappa = {res['kappa_all']:.3f} | agreement {agree}/{len(pk)} = {agree/len(pk):.1%} | n>=3 subset kappa {res['kappa_n3']:.3f} | disagreements {len(res['disagreements'])} (in A's alt_tags: {alt_ok}, A-uncertain: {unc})")
json.dump(pk, open(f'{ROOT}/eval/pk_evalset_extracted.json', 'w'), ensure_ascii=False, indent=1)
json.dump(res, open(f'{ROOT}/results_m/m4_kappa.json', 'w'), indent=1); print('updated eval/pk_evalset_extracted.json (annotator_b_tag) and results_m/m4_kappa.json')

In [ ]:
# L0.3 — write the adjudication sheet: the disagreements, for the author to decide the final tag
out_x = f'{ROOT}/eval/pk_adjudication_sheet.xlsx'
if os.path.exists(out_x):
    print(f'{out_x} already exists — not overwriting (it holds the filled adjudication). Delete it first to regenerate.')
else:
    kap = json.load(open(f'{ROOT}/results_m/m4_kappa.json'))
    adj = pd.DataFrame([[d['row'], d['sentence'], d['A'], d['B'], ', '.join(d['alt']) or '', 'y' if d['uncertain'] else '', ''] for d in kap['disagreements']],
                       columns=['row', 'sentence', 'label_A_collection', 'label_B_author', 'A_alternatives', 'A_uncertain', 'final_tag'])
    with pd.ExcelWriter(out_x) as xw:
        adj.to_excel(xw, sheet_name='adjudicate', index=False); pd.DataFrame({'tag': sorted(LEX)}).to_excel(xw, sheet_name='tag_list', index=False)
    adj.to_csv(f'{ROOT}/eval/pk_adjudication_sheet.csv', index=False)
    print(f'wrote eval/pk_adjudication_sheet.xlsx / .csv ({len(adj)} rows) — fill final_tag')

In [ ]:
# L0.4 — apply the author's adjudication: final_tag becomes gold_tag (collection label kept as collection_tag). Run after eval/pk_adjudication_sheet.xlsx is filled.
import json, os, pandas as pd
from collections import Counter
LEX = set(json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon'])
p_x, p_c = f'{ROOT}/eval/pk_adjudication_sheet.xlsx', f'{ROOT}/eval/pk_adjudication_sheet.csv'
if os.path.getmtime(p_x) > os.path.getmtime(p_c):
    book = pd.read_excel(p_x, sheet_name=None); ad = next(df for df in book.values() if {'row', 'final_tag'} <= set(map(str, df.columns)))
else: ad = pd.read_csv(p_c)
ad['final_tag'] = ad['final_tag'].astype(str).str.strip().str.lower().replace({'nan': ''})
empty = ad[ad.final_tag == '']; bad = ad[(ad.final_tag != '') & ~ad.final_tag.isin(LEX)]
assert len(empty) == 0, f'{len(empty)} rows still undecided: {empty.row.tolist()}'
assert len(bad) == 0, f'tags outside the list: {bad[["row", "final_tag"]].values.tolist()}'
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); F = dict(zip(ad.row.astype(int), ad.final_tag)); changed = []
for r in pk:
    r.setdefault('collection_tag', r['gold_tag'])
    if r['row'] in F and F[r['row']] != r['gold_tag']: changed.append((r['row'], r['gold_tag'], F[r['row']])); r['gold_tag'] = F[r['row']]
json.dump(pk, open(f'{ROOT}/eval/pk_evalset_extracted.json', 'w'), ensure_ascii=False, indent=1)
d = Counter(r['gold_tag'] for r in pk); print(f'adjudicated {len(F)} rows, {len(changed)} gold tags changed | now {len(d)} tags, n>=3 tags: {sum(c >= 3 for c in d.values())} covering {sum(c for c in d.values() if c >= 3)} rows')
print('changed:', changed)
print('NOTE: the Pakistan evaluation cells (M5.2 / M5.3 / Q7b / P1.5 / P2.3–P2.6 in the experiments notebook, or the re-evaluation cell below) must be re-run so every Pakistan number reflects the adjudicated labels.')

### How to fill the annotation sheet (author)

**File:** `MyDrive/geoproject/eval/pk_annotation_sheet.xlsx` (sheet `annotate`; the 227 allowed tags are on sheet `tag_list`). The `.csv` next to it is the same thing if you prefer a text editor — fill only one of the two.

**Rules — the same protocol the base paper used:**
1. Read **only the `sentence`**. Do not look up the venue, the city, or the earlier labels — the point is an independent pass.
2. Put exactly **one** tag in `tag`, spelled exactly as in `tag_list` (lower-case, hyphens: `fast-food`, `city-gate`, `cave-entrance`). Type the tag as it appears in the list; do not invent new tags.
3. If two tags are both plausible, put the more likely one in `tag` and the other in `alt_tag`; put `y` in `unsure`. Otherwise leave `alt_tag` and `unsure` empty.
4. Pakistani usage to keep in mind: a "hotel" that serves food is a `restaurant`; dhaba / karahi / nihari place → `restaurant` (or `fast-food` for stalls, rolls, fried snacks); bazaar / mandi → `marketplace`; masjid → `mosque`; mazar / darbar / dargah → `shrine`; qila → `fort`; bagh → `garden` or `park`; chowk → `square`.
5. Work through the sheet in order (it is already shuffled). Don't go back and revise earlier rows after you've seen later ones — first judgement is what we want. ~199 rows ≈ 1.5–2 h.
6. Save the file in place (same name, same folder). Then run the cell **L0.2** above: it checks every tag is in the list, computes Cohen's κ against the collection-time labels, and stores your labels as `annotator_b_tag` in the evaluation file.

After L0.2, the disagreement list it prints is yours to resolve: for each row decide the final `gold_tag` (a later cell will apply your decisions).

In [ ]:
# Helpers — one style, one palette, one save path for every figure and table; the index is written at the end
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, pandas as pd, numpy as np, json, os, re, textwrap
plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.6, 'axes.edgecolor': '#c3c2b7', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
                     'legend.frameon': False, 'legend.fontsize': 8, 'figure.dpi': 110, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42})
C = {'prev': '#898781',   # previous approach / base paper (neutral reference)
     'ours': '#2a78d6',   # this work (adjective->tag blend)
     'ours2': '#eb6834',  # this work, second variant (Bayesian / best backbone)
     'naive': '#1baf7a',  # naive adjectives (dilution control) — always direct-labelled
     'ink': '#0b0b0b', 'muted': '#898781', 'grid': '#e1e0d9', 'base': '#c3c2b7'}
SEQ = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#1c5cab', '#0d366b']      # one-hue sequential ramp (heatmaps)
pd.set_option('styler.latex.hrules', True)
def fig(w=6.3, h=3.4, **kw): return plt.subplots(figsize=(w, h), **kw)
def _fn(kind, iid, stage): return f'{kind}_{iid.replace(".", "-")}_{stage}'
def _idx(entry): INDEX[:] = [e for e in INDEX if e[0] != entry[0]] + [entry]     # one entry per id (re-running a cell replaces it)
def save_fig(f, iid, stage, caption, source):
    fn = _fn('fig', iid, stage); f.savefig(f'{FIG}/{fn}.pdf', bbox_inches='tight'); f.savefig(f'{FIG}/{fn}.png', bbox_inches='tight'); plt.close(f)
    _idx((f'F{iid}', f'figures/{fn}.pdf', stage, caption, source)); print('saved', fn)
def save_tab(df, iid, stage, caption, source, float_fmt='%.3f', col_width=None):
    fn = _fn('tab', iid, stage); df.to_csv(f'{TAB}/{fn}.csv', index=False)
    open(f'{TAB}/{fn}.tex', 'w').write(df.to_latex(index=False, float_format=float_fmt, escape=True, caption=caption, label=f'tab:{fn}'))
    show = df.copy()
    for c in show.columns:
        if show[c].dtype.kind == 'f': show[c] = show[c].map(lambda v: '' if pd.isna(v) else float_fmt % v)
        show[c] = show[c].astype(str).map(lambda s: textwrap.fill(s, col_width or 38))
    nrow, ncol = show.shape; f, ax = plt.subplots(figsize=(min(1.3 * ncol + 1, 11), 0.32 * nrow + 0.8)); ax.axis('off')
    t = ax.table(cellText=show.values, colLabels=show.columns, loc='center', cellLoc='left', colLoc='left'); t.auto_set_font_size(False); t.set_fontsize(7); t.scale(1, 1.25)
    for (r, c), cell in t.get_celld().items():
        cell.set_edgecolor('#e1e0d9'); cell.set_linewidth(0.5)
        if r == 0: cell.set_text_props(weight='bold'); cell.set_facecolor('#f0efec')
    f.savefig(f'{TAB}/{fn}.png', bbox_inches='tight'); plt.close(f)
    _idx((f'T{iid}', f'tables/{fn}.csv', stage, caption, source)); print('saved', fn)
def _read_index():                      # so a partial re-run cannot wipe entries written in an earlier session
    p = f'{FIG}/INDEX.md'
    if not os.path.exists(p): return []
    out = []
    for l in open(p).read().splitlines():
        if not l.startswith('| ') or l.startswith('| id ') or set(l) <= set('|- '): continue
        c = [x.strip() for x in l.strip('|').split('|')]
        if len(c) >= 5: out.append((c[0], c[1].strip('`'), c[2], c[3], c[4].strip('`')))
    return out
def write_index():
    merged = {e[0]: e for e in _read_index()}; merged.update({e[0]: e for e in INDEX}); INDEX[:] = list(merged.values())
    items = sorted(INDEX, key=lambda e: (int(re.match(r'[FT](\d)\.(\d+)', e[0]).group(1)), int(re.match(r'[FT](\d)\.(\d+)', e[0]).group(2)), e[0]))
    rows = ['| id | file | stage | caption (draft) | generated from |', '|---|---|---|---|---|'] + [f'| {i} | `{f}` | {s} | {c} | `{src}` |' for i, f, s, c, src in items]
    open(f'{FIG}/INDEX.md', 'w').write('# Figures and tables index\n\n' + '\n'.join(rows) + '\n'); print(f'INDEX.md: {len(items)} items')
print('helpers ready; palette:', {k: v for k, v in C.items() if k in ('prev', 'ours', 'ours2', 'naive')})

In [ ]:
# Shared — the ranker (used by the worked example, Hits@k curves, per-tag and explanation items). Same code path as the experiments; nothing re-fitted here.
import pickle, re, json, numpy as np
from itertools import groupby
from sklearn.model_selection import KFold
LEX = json.load(open(f'{ROOT}/eval/evalset_106.json'))['lexicon']
ABSTRACT = {'place','spot','location','area','site','position','thing','kind','sort','lot','bit','way','day','time'}
KS, LAMS, LAM = (1, 2, 3, 5, 10, 20), [0.0, 0.1, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5], 0.35
def parts(s): return [w for w in re.split(r'[\s\-_/]+', s) if w]
def variants(s):
    b = [s, s.lower(), s.title(), s.capitalize()]; o, seen = [], set()
    for v in b + [x.replace(' ', '_').replace('-', '_') for x in b] + [x.replace(' ', '').replace('-', '') for x in b]:
        if v and v not in seen: seen.add(v); o.append(v)
    return o
def ladder(VEC):
    def f(s):
        for v in variants(s):
            if v in VEC: return VEC[v]
        sub = [VEC[v] for w in parts(s) for v in variants(w) if v in VEC][:8]
        return np.mean(sub, 0) if sub else None
    return f
VECS = {'W2V': ladder(pickle.load(open(f'{ROOT}/models_cache/w2v_googlenews_subset_v2.pkl', 'rb'))), 'Numberbatch': ladder(pickle.load(open(f'{ROOT}/models_cache/numberbatch_subset.pkl', 'rb')))}
enz = np.load(f'{ROOT}/results_i/i3_elasticnet.npz', allow_pickle=True); COEF, EN_CLS, EN_ADJ = enz['coef'], list(enz['classes']), list(enz['adj_vocab']); EN_AIX = {a: j for j, a in enumerate(EN_ADJ)}
ben = np.load(f'{ROOT}/results_p/p1_bayesian_en.npz', allow_pickle=True); meta = np.load(f'{ROOT}/results_p/p1_feat_meta.npz', allow_pickle=True)
PV, PCLS, NADJ = list(meta['patvocab']), list(meta['classes']), int(meta['n_adj']); PIX = {p: j for j, p in enumerate(PV)}; ADJ_SET, PAIR_SET = set(PV[:NADJ]), set(PV[NADJ:])
BMU, BSEL, BLO, BHI = ben['mean'], ben['selected'], ben['hdi_lo'], ben['hdi_hi']
def pat_feats(r): return [PIX[a] for a in r['adjectives'] if a in ADJ_SET] + [PIX[p] for p in r['transaction'] if p in PAIR_SET]
class Ranker:
    def __init__(self, name):
        vec = VECS[name]; self.name = name; self.TAGS = [t for t in LEX if vec(t) is not None]; self.IX = {t: i for i, t in enumerate(self.TAGS)}
        M = np.vstack([vec(t) for t in self.TAGS]); self.M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9); self.vec = vec
        self.c2w_adj = {c: self.IX[c] for c in EN_CLS if c in self.IX}; self.c2w_pat = {c: self.IX[c] for c in PCLS if c in self.IX}
    def sim(self, terms):
        s = np.zeros(len(self.TAGS), np.float32)
        for t in terms:
            v = self.vec(t)
            if v is None: continue
            sims = self.M @ (v / (np.linalg.norm(v) + 1e-9)); s += (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
        return s
    def _norm(self, s): lo, hi = s.min(), s.max(); return (s - lo) / (hi - lo + 1e-9) if hi > lo else s
    def en_adj(self, r):                       # adjective-level elastic net (the headline blend)
        s = np.zeros(len(self.TAGS), np.float32); rc = [COEF[:, EN_AIX[a]] for a in r['adjectives'] if a in EN_AIX]
        if not rc: return s
        agg = np.sum(rc, 0)
        for c, w in self.c2w_adj.items(): s[w] = agg[EN_CLS.index(c)]
        return self._norm(s)
    def en_ben(self, r):                       # Bayesian elastic net over FP-Growth patterns (HDI-selected posterior means)
        s = np.zeros(len(self.TAGS), np.float32); f = pat_feats(r)
        if not f: return s
        agg = (BMU * BSEL)[f].sum(0)
        for c, w in self.c2w_pat.items(): s[w] = agg[PCLS.index(c)]
        return self._norm(s)
    def scores(self, r, lam=LAM, en='adj', naive=False):
        terms = [t for t in r['context_nouns'] if t.lower() not in ABSTRACT] + (r['adjectives'] if naive else [])
        s = self._norm(self.sim(terms))
        if lam > 0 and en: s = (1 - lam) * s + lam * (self.en_adj(r) if en == 'adj' else self.en_ben(r))
        return s
    def rank(self, r, **kw):
        s = self.scores(r, **kw); g = r['gold_tag']; return int((s >= s[self.IX[g]]).sum()) if g in self.IX else len(self.TAGS) + 1
    def ranks(self, rows, **kw): return np.array([self.rank(r, **kw) for r in rows], float)
def metrics(ranks, ks=KS):
    r = np.asarray(ranks, float); o = {f'Hits@{k}': float((r <= k).mean()) for k in ks}; o['nDCG@20'] = float(np.where(r <= 20, 1 / np.log2(r + 1), 0).mean()); o['MedR'] = float(np.median(r)); return o
def cv_ranks(rk, rows, **kw):                  # lambda chosen per fold on the other folds (argmax train-fold Hits@3)
    R = {L: rk.ranks(rows, lam=L, **kw) for L in LAMS}; held = np.zeros(len(rows)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return held, ch
RK = {n: Ranker(n) for n in VECS}
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
dpk = Counter(r['gold_tag'] for r in pk); SUB = np.array([dpk[r['gold_tag']] >= 3 for r in pk])
chk = metrics(RK['W2V'].ranks(ev106)); print(f"ranker check (W2V + adjective table, λ=.35 on the base-paper set): Hits@1 {chk['Hits@1']:.3f} Hits@3 {chk['Hits@3']:.3f} nDCG {chk['nDCG@20']:.3f}  <- must equal the experiment values")

In [ ]:
# Re-evaluation on the adjudicated Pakistan labels — rescores every Pakistan arm with the saved models and rewrites the result files the tables read.
# (Global-set entries are untouched: their labels did not change.)  Sets label_version in each file.
import time, torch, pickle, numpy as np, json
from scipy import sparse
from sklearn.linear_model import LogisticRegression
LABEL_VERSION = 'adjudicated (collection labels + author pass, disagreements resolved by the author)'
pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')); dpk = Counter(r['gold_tag'] for r in pk); SUB = np.array([dpk[r['gold_tag']] >= 3 for r in pk]); N3 = int(SUB.sum())
def en_from(rk, coef, classes, vocab):            # adjective-level table with arbitrary coefficients, aligned to a ranker's tag space
    aix = {a: j for j, a in enumerate(vocab)}; cix = {c: k for k, c in enumerate(classes)}; c2w = {c: rk.IX[c] for c in classes if c in rk.IX}
    def en(r):
        s = np.zeros(len(rk.TAGS), np.float32); rc = [coef[:, aix[a]] for a in r['adjectives'] if a in aix]
        if not rc: return s
        agg = np.sum(rc, 0)
        for c, w in c2w.items(): s[w] = agg[cix[c]]
        return rk._norm(s)
    return en
def pen_from(rk, coefPC, cols=None):               # pattern-level table (P x C) aligned to a ranker's tag space
    cix = {c: k for k, c in enumerate(PCLS)}; c2w = {c: rk.IX[c] for c in PCLS if c in rk.IX}; colset = None if cols is None else set(cols.tolist())
    def en(r):
        s = np.zeros(len(rk.TAGS), np.float32); f = [j for j in pat_feats(r) if colset is None or j in colset]
        if not f: return s
        agg = coefPC[f].sum(0)
        for c, w in c2w.items(): s[w] = agg[cix[c]]
        return rk._norm(s)
    return en
def ranks_with(rk, rows, en=None, lam=LAM, naive=False, clustered=False):
    out = []
    for r in rows:
        terms = [t for t in r['context_nouns'] if t.lower() not in ABSTRACT] + (r['adjectives'] if naive else [])
        s = rk._norm(rk.sim(terms))
        if lam > 0 and en is not None: s = (1 - lam) * s + lam * en(r)
        g = r['gold_tag']
        if clustered:
            ranked = [rk.TAGS[i] for i in np.argsort(-s, kind='stable')]; pr = ranked[:20]; ps = set(pr); S = rk.M[[rk.IX[t] for t in pr]] @ rk.M.T
            for i, t in enumerate(pr): S[i, rk.IX[t]] = -np.inf
            votes = Counter({t: 0 for t in pr})
            for i in range(len(pr)):
                for j in np.argsort(-S[i])[:4]:
                    if rk.TAGS[j] in ps: votes[rk.TAGS[j]] += 1
            seen = 0; rank = len(rk.TAGS) + 1
            for _, grp in groupby(sorted(votes.items(), key=lambda kv: -kv[1]), key=lambda kv: kv[1]):
                cl = [t for t, _ in grp]; seen += len(cl)
                if g in cl: rank = seen; break
            out.append(rank if g in ps else len(rk.TAGS) + 1)
        else: out.append(int((s >= s[rk.IX[g]]).sum()) if g in rk.IX else len(rk.TAGS) + 1)
    return np.array(out, float)
def cv_with(rk, rows, en, **kw):
    R = {L: ranks_with(rk, rows, en, lam=L, **kw) for L in LAMS}; held = np.zeros(len(rows)); ch = []
    for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(rows))):
        L = max(LAMS, key=lambda l: (R[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = R[L][te_i]
    return held, ch
both = lambda rk_: {'full_199': metrics(rk_), 'n3_158': metrics(rk_[SUB])}
W = RK['W2V']; EN1 = W.en_adj
# --- m5_transfer.json / m5_region_experiments.json ---
maj = dpk.most_common(1)[0][0]
R = {'majority class': np.array([1.0 if r['gold_tag'] == maj else len(W.TAGS) + 1 for r in pk]), 'freq prior (oracle)': np.array([sum(c >= dpk[r['gold_tag']] for c in dpk.values()) for r in pk], float),
     'W2V (Alg.2)': ranks_with(W, pk, None, lam=0), 'W2V-Clustered (Alg.3)': ranks_with(W, pk, None, lam=0, clustered=True), 'W2V plain + abstract rm': ranks_with(W, pk, None, lam=0), 'W2V-Clust + abstract rm': ranks_with(W, pk, None, lam=0, clustered=True),
     'transfer v1, λ=0.35 fixed': ranks_with(W, pk, EN1)}
h, ch = cv_with(W, pk, EN1); R['transfer v1, λ by 5-fold CV'] = h; cvl = {'transfer': ch}
auto = lambda r: {**r, 'context_nouns': r['ours_nouns']}
R['[auto nouns] W2V plain'] = ranks_with(W, [auto(r) for r in pk], None, lam=0); R['[auto nouns] transfer λ=0.35'] = ranks_with(W, [auto(r) for r in pk], EN1)
lm = np.load(f'{ROOT}/results_m/m5_en_local_merged.npz', allow_pickle=True)
ENS = {'local-only': en_from(W, lm['coef_local'], list(lm['classes_local']), list(lm['vocab_local'])), 'merged global+PK': en_from(W, lm['coef_merged'], EN_CLS, list(lm['vocab_v1'])), 'merged, PK x10': en_from(W, lm['coef_merged_pk10'], EN_CLS, list(lm['vocab_v1']))}
for nm, en in ENS.items():
    R[f'{nm}, λ=0.35'] = ranks_with(W, pk, en); h, ch = cv_with(W, pk, en); R[f'{nm}, λ by CV'] = h; cvl[nm] = ch
out = {'full_199': {k: metrics(v) for k, v in R.items()}, 'n3_158': {k: metrics(v[SUB]) for k, v in R.items()}, 'n_n3': N3, 'cv_lambdas': cvl, 'label_version': LABEL_VERSION}
json.dump({**out, 'transfer_cv_lambdas': cvl['transfer']}, open(f'{ROOT}/results_m/m5_transfer.json', 'w'), indent=1); json.dump(out, open(f'{ROOT}/results_m/m5_region_experiments.json', 'w'), indent=1)
print(f"Pakistan n>=3 ({N3}): base {out['n3_158']['W2V (Alg.2)']['Hits@3']:.3f} | transfer {out['n3_158']['transfer v1, λ=0.35 fixed']['Hits@3']:.3f} | PKx10 CV {out['n3_158']['merged, PK x10, λ by CV']['Hits@3']:.3f}")
# --- m5_transfer_v2.json (corpus variant) ---
q7 = np.load(f'{ROOT}/results_q/q7_v2_en.npz', allow_pickle=True); EN2 = en_from(W, q7['coef'], list(q7['classes']), list(q7['adj_vocab']))
R2 = {'transfer v1, λ=0.35': R['transfer v1, λ=0.35 fixed'], 'transfer v2, λ=0.35': ranks_with(W, pk, EN2)}; h, ch = cv_with(W, pk, EN2); R2['transfer v2, λ by CV'] = h; R2['[auto nouns] transfer v2 λ=0.35'] = ranks_with(W, [auto(r) for r in pk], EN2)
json.dump({'full_199': {k: metrics(v) for k, v in R2.items()}, 'n3_158': {k: metrics(v[SUB]) for k, v in R2.items()}, 'label_version': LABEL_VERSION}, open(f'{ROOT}/results_m/m5_transfer_v2.json', 'w'), indent=1)
# --- m5_leakage_robust.json ---
lr = json.load(open(f'{ROOT}/results_m/m5_leakage_robust.json')); leak = set(lr['excluded_rows']); clean = np.array([r['row'] not in leak for r in pk])
json.dump({'excluded_rows': sorted(leak), 'n_clean_n3': int((clean & SUB).sum()), 'W2V_clean': metrics(R['W2V (Alg.2)'][SUB & clean]), 'transfer_clean': metrics(R['transfer v1, λ=0.35 fixed'][SUB & clean]), 'label_version': LABEL_VERSION}, open(f'{ROOT}/results_m/m5_leakage_robust.json', 'w'), indent=1)
# --- p2_backbones.json (Pakistan entries) ---
p2 = json.load(open(f'{ROOT}/results_p/p2_backbones.json'))
CACHE = {'W2V (GoogleNews)': 'w2v_googlenews_subset_v2.pkl', 'GloVe (wiki-gigaword)': 'glove_subset.pkl', 'fastText (wiki-news)': 'fasttext_subset.pkl', 'ConceptNet Numberbatch': 'numberbatch_subset.pkl', 'all-mpnet-base-v2': 'mpnet_terms.pkl', 'bge-base-en-v1.5': 'bge_terms.pkl', 'multilingual-e5-base': 'e5_terms.pkl'}
for b, fn in CACHE.items():
    VEC = pickle.load(open(f'{ROOT}/models_cache/{fn}', 'rb')); VECS[b] = ladder(VEC) if fn.endswith('_subset.pkl') else VEC.get; rk = Ranker(b)
    p2[b]['nouns']['PK n>=3'] = {**metrics(ranks_with(rk, pk, None, lam=0)[SUB]), 'cv_lambdas': None}; p2[b]['naive']['PK n>=3'] = {**metrics(ranks_with(rk, pk, None, lam=0, naive=True)[SUB]), 'cv_lambdas': None}
    h, ch = cv_with(rk, pk, rk.en_adj); p2[b]['en']['PK n>=3'] = {**metrics(h[SUB]), 'cv_lambdas': ch}
p2['label_version'] = LABEL_VERSION; json.dump(p2, open(f'{ROOT}/results_p/p2_backbones.json', 'w'), indent=1); print('backbones PK Hits@3:', {b: round(p2[b]['en']['PK n>=3']['Hits@3'], 3) for b in CACHE})
# --- p1_ben_figure2.json (pattern-level blends, Pakistan entries) ---
fbj = json.load(open(f'{ROOT}/results_p/p1_ben_figure2.json')); FC = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True)['coef'].T
E = fbj['eval']['PK n>=3'] = {'nouns_only': metrics(ranks_with(W, pk, None, lam=0)[SUB])}
for name, en in [('freq EN (966 patterns)', pen_from(W, FC)), ('Bayesian EN, HDI-selected', pen_from(W, BMU * BSEL)), ('Bayesian EN, posterior mean', pen_from(W, BMU))]:
    h, ch = cv_with(W, pk, en); E[name] = {'cv': metrics(h[SUB]), 'folds': ch, 'lam035': metrics(ranks_with(W, pk, en)[SUB])}
fbj['label_version'] = LABEL_VERSION; json.dump(fbj, open(f'{ROOT}/results_p/p1_ben_figure2.json', 'w'), indent=1); print('pattern-level PK Hits@3:', {k: round(v['cv']['Hits@3'], 3) for k, v in E.items() if k != 'nouns_only'})
# --- p2_granularity.json (Pakistan entries; refit the three family models on the same 120k subsample) ---
Xtr = sparse.load_npz(f'{ROOT}/results_p/Xtr.npz'); ytr = meta['ytr']; rng = np.random.default_rng(42)
keep = np.concatenate([rng.choice(np.where(ytr == c)[0], min((ytr == c).sum(), max(1, round((ytr == c).mean() * 120_000))), replace=False) for c in range(len(PCLS))])
g = json.load(open(f'{ROOT}/results_p/p2_granularity.json')); FAM = {'adjectives only (1-itemsets)': np.arange(NADJ), 'adj-noun pairs only (2-itemsets)': np.arange(NADJ, len(PV)), 'both (P1.3)': np.arange(len(PV))}
for fam, cols in FAM.items():
    t0 = time.time(); m_ = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.5, max_iter=200, tol=1e-3, n_jobs=-1).fit(Xtr[keep][:, cols], ytr[keep])
    coefPC = np.zeros((len(PV), len(PCLS)), np.float32); coefPC[cols] = m_.coef_.T; en = pen_from(W, coefPC, cols); h, ch = cv_with(W, pk, en)
    g[fam]['PK n>=3'] = {'coverage': float(np.mean([any(j in set(cols.tolist()) for j in pat_feats(r)) for r, k in zip(pk, SUB) if k])), **metrics(h[SUB]), 'cv_lambdas': ch}; print(f'  granularity {fam}: refit {time.time()-t0:.0f}s, PK Hits@3 {g[fam]["PK n>=3"]["Hits@3"]:.3f}')
g['label_version'] = LABEL_VERSION; json.dump(g, open(f'{ROOT}/results_p/p2_granularity.json', 'w'), indent=1)
# --- p2_classifier.json (Pakistan entry; saved DistilBERT, CPU) ---
from transformers import AutoTokenizer, AutoModelForSequenceClassification
pc = json.load(open(f'{ROOT}/results_p/p2_classifier.json')); tok = AutoTokenizer.from_pretrained(f'{ROOT}/models_cache/distilbert_weak69'); mdl = AutoModelForSequenceClassification.from_pretrained(f'{ROOT}/models_cache/distilbert_weak69').eval()
CLS = sorted(pd.read_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet', columns=['weak_tag']).weak_tag.unique()); CIX = {c: i for i, c in enumerate(CLS)}
with torch.no_grad(): P = np.vstack([torch.softmax(mdl(**tok([r['sentence'] for r in pk[i:i+64]], padding=True, truncation=True, max_length=128, return_tensors='pt')).logits.float(), -1).numpy() for i in range(0, len(pk), 64)])
r0 = np.array([int((p >= p[CIX[r['gold_tag']]]).sum()) if r['gold_tag'] in CIX else len(CLS) + 1 for r, p in zip(pk, P)], float)
def fuse(lam):
    out = []
    for r, p in zip(pk, P):
        s = np.zeros(len(W.TAGS), np.float32)
        for c, k in CIX.items():
            if c in W.IX: s[W.IX[c]] = p[k]
        s = W._norm(s); s = (1 - lam) * s + lam * EN1(r); g_ = r['gold_tag']; out.append(int((s >= s[W.IX[g_]]).sum()) if g_ in W.IX else len(W.TAGS) + 1)
    return np.array(out, float)
Rf = {L: fuse(L) for L in LAMS}; held = np.zeros(len(pk)); ch = []
for tr_i, te_i in KFold(5, shuffle=True, random_state=42).split(np.arange(len(pk))):
    L = max(LAMS, key=lambda l: (Rf[l][tr_i] <= 3).mean()); ch.append(L); held[te_i] = Rf[L][te_i]
pc['PK n>=3'] = {'gold_coverage_69': float(np.mean([r['gold_tag'] in CIX for r, k in zip(pk, SUB) if k])), 'classifier_only': metrics(r0[SUB]), 'classifier_plus_en_cv': metrics(held[SUB]), 'cv_lambdas': ch}; pc['label_version'] = LABEL_VERSION
json.dump(pc, open(f'{ROOT}/results_p/p2_classifier.json', 'w'), indent=1); print('classifier PK Hits@3:', round(pc['PK n>=3']['classifier_only']['Hits@3'], 3), '| + fusion', round(pc['PK n>=3']['classifier_plus_en_cv']['Hits@3'], 3))
print('all Pakistan result files rewritten on the adjudicated labels')

## Stage 1 — Data: sources, corpora and evaluation sets (Ch. 3)
Sources table, collection funnel, corpus composition, sample rows, evaluation-set statistics and tag distributions, the Pakistan map, annotation agreement, and the train/test overlap check.

In [ ]:
# T3.1 — data sources · F3.1 — collection funnel (re-export) · F3.2 — corpus composition
import shutil, pyarrow.parquet as pq
cc = json.load(open(f'{ROOT}/results_i/corpus_composition.json')); prov = json.load(open(f'{ROOT}/dataset_provenance.json'))
n_wd = pq.ParquetFile(f'{ROOT}/raw_q/wikidata_places.parquet').metadata.num_rows
n_wl = pq.ParquetFile(f'{ROOT}/results_i/weak_labelled.parquet').metadata.num_rows
pk_n = len(json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))); ev_n = len(json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')))
t31 = pd.DataFrame([
    ['English Wikivoyage (dump)', prov['license'], f"snapshot {prov['snapshot']} ({prov['download_date']})", cc['by_source_raw']['wikivoyage'], cc['by_source_corpus_v1']['wikivoyage'], 'training corpus (weak labels, pattern mining)'],
    ['Wikimedia Commons, Pakistan captions', 'CC (per file)', 'API, 2026-07', cc['by_source_raw']['commons_pk'], cc['by_source_corpus_v1']['commons_pk'], 'training corpus (Pakistan)'],
    ['OpenStreetMap descriptions, Pakistan', 'ODbL', 'Overpass, 2026-07', cc['by_source_raw']['osm_pk'], cc['by_source_corpus_v1']['osm_pk'], 'training corpus (Pakistan)'],
    ['Wikidata place descriptions (P31 typed)', 'CC0', 'QLever SPARQL, 2026-09', n_wd, n_wd, 'corpus-variant experiments only'],
    ['Base-paper evaluation set (Yousaf et al. 2023)', 'CC0 (published)', 'github: place-descriptions', ev_n, ev_n, 'evaluation (global)'],
    ['Pakistan evaluation set (this work)', 'CC BY-SA 4.0 / ODbL / Reddit user content', 'collected 2026-09', pk_n, pk_n, 'evaluation (local)']],
    columns=['Source', 'Licence', 'Snapshot / access', 'Raw rows', 'Rows used', 'Role'])
save_tab(t31, '3.1', 'data_sources', 'Data sources, licences, snapshot dates and row counts. "Rows used" for the training corpora is after language routing, near-duplicate removal and quality gating.', 'results_i/corpus_composition.json, dataset_provenance.json, raw_q/wikidata_places.parquet, eval/*')
# F3.1 — the funnel already exists as a generated figure; re-export it under the chapter naming
for ext in ('pdf', 'png'): shutil.copy(f'{ROOT}/figures_blockd/01_pipeline.{ext}', f'{FIG}/fig_3-1_collection_funnel.{ext}')
INDEX.append(('F3.1', 'figures/fig_3-1_collection_funnel.pdf', 'data_sources', 'Collection funnel from raw Wikivoyage/Commons/OSM records to the annotated training corpus; every count is read from the corresponding parquet file.', 'figures_blockd/01_pipeline (generated in the collection notebook)'))
# F3.2 — composition, three small panels (share of source / anchor quality / language)
f, ax = fig(7.2, 2.4, ncols=3)
def hbar(a, labels, vals, title, color):
    vals = np.array(vals, float); order = np.argsort(vals); a.barh(np.array(labels)[order], vals[order], color=color, height=0.55)
    for y, v in enumerate(vals[order]): a.text(v, y, f' {v:,.0f}' if v >= 1 else f' {v:.1%}', va='center', fontsize=7.5, color=C['ink'])
    a.set_title(title, loc='left'); a.set_xscale('log'); a.grid(axis='y', visible=False); a.set_xlim(right=a.get_xlim()[1] * 8)
hbar(ax[0], ['Wikivoyage', 'Commons (Pakistan)', 'OSM (Pakistan)'], list(cc['by_source_raw'].values()), 'Raw records by source', C['ours'])
hbar(ax[1], ['page centroid', 'listing', 'element'], list(cc['by_anchor_quality_corpus_v1'].values()), 'Corpus rows by coordinate anchor', C['ours'])
hbar(ax[2], ['routed English', 'held: unclassified', 'of which Urdu script'], [cc['routed_english'], cc['held_unclassified'], cc['held_urdu_script']], 'Sentences by language routing', C['ours'])
f.tight_layout(); save_fig(f, '3.2', 'corpus_composition', f'Composition of the training corpus by source, coordinate-anchor quality and language routing (log scale). 74% of corpus rows inherit a page-level centroid. The language router held out {cc["held_unclassified"]} sentences it could not classify as English; an audit (Section 4.8) found only {cc["held_urdu_script"]} of them are Urdu script, the rest being English text from non-English-speaking destinations. The corpus therefore contains essentially no Urdu or Roman-Urdu material, which is why the multilingual evaluation uses hand-written translations.', 'results_i/corpus_composition.json')

In [ ]:
# T3.2 — sample rows from each dataset · T3.3 — evaluation-set statistics · F3.3 — gold-tag distributions
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
wl = pd.read_parquet(f'{ROOT}/results_m/weak_labelled_v2.parquet', columns=['text', 'weak_tag', 'transaction', 'region', 'split'])
wd = pd.read_parquet(f'{ROOT}/raw_q/wikidata_places.parquet').head(5000)
txt_col = 'description' if 'description' in wd.columns else max((c for c in wd.columns if wd[c].dtype == object), key=lambda c: wd[c].astype(str).str.len().mean())
tag_col = next(c for c in ('tag', 'label', 'osm_tag') if c in wd.columns)
rng = np.random.default_rng(7); short = lambda s, n=110: (s[:n] + '…') if len(s) > n else s
def ex_eval(rows, name):
    pick = [rows[i] for i in rng.choice(len(rows), 4, replace=False)]
    return [[name, short(r['sentence']), ', '.join(r['context_nouns']), ', '.join(r['adjectives']), r['gold_tag']] for r in pick]
wl_s = wl[(wl.split == 'train') & wl.transaction.str.contains('amod:')].sample(4, random_state=7)
wd_s = wd[wd[txt_col].astype(str).str.len() > 30].sample(4, random_state=7)
t32 = pd.DataFrame(ex_eval(ev106, 'Base-paper evaluation set') + ex_eval(pk, 'Pakistan evaluation set')
                   + [['Weakly-labelled training corpus', short(r.text), '', ', '.join(x[5:].replace('_', ' ') for x in json.loads(r.transaction) if x.startswith('amod:'))[:80], r.weak_tag] for r in wl_s.itertuples()]
                   + [['Wikidata descriptions', short(str(r[txt_col])), '', '', str(r[tag_col])] for _, r in wd_s.iterrows()],
                   columns=['Dataset', 'Text', 'Context nouns', 'Adjectives / adjective–noun pairs', 'Tag'])
save_tab(t32, '3.2', 'sample_rows', 'Sample rows from each dataset: the two evaluation sets (with the extracted context nouns and adjectives), the weakly-labelled training corpus (with its mined adjective–noun pairs and weak tag) and the Wikidata descriptions used in the corpus-variant experiments.', 'eval/evalset_106_extracted.json, eval/pk_evalset_extracted.json, results_m/weak_labelled_v2.parquet, raw_q/wikidata_places.parquet', col_width=48)

# T3.3 — statistics side by side (urban / rural split of the Pakistan set by city field)
URBAN = {'Karachi', 'Islamabad', 'Lahore', 'Rawalpindi', 'Multan', 'Peshawar', 'Faisalabad', 'Quetta', 'Hyderabad', 'Sialkot', 'Gujranwala', 'Bahawalpur', 'Sukkur', 'Abbottabad', 'Mardan', 'Sargodha'}
pk_city = [str(r.get('city') or '') for r in pk]; urban = np.array([c in URBAN for c in pk_city])
def stats(rows, name, extra=None):
    d = Counter(r['gold_tag'] for r in rows); n = len(rows); top = sorted(d.values(), reverse=True)
    o = {'Evaluation set': name, 'Sentences': n, 'Distinct tags': len(d), 'Singleton tags': sum(1 for c in d.values() if c == 1), 'Tags with n≥3': sum(1 for c in d.values() if c >= 3),
         'Rows in n≥3 tags': sum(c for c in d.values() if c >= 3), 'Top-5 tags': ', '.join(f'{t} ({c})' for t, c in d.most_common(5)), 'Majority share (= Hits@1 floor)': round(top[0] / n, 3),
         'Frequency-prior Hits@3 (oracle)': round(sum(top[:3]) / n, 3), 'Adjectives per sentence': round(float(np.mean([len(r['adjectives']) for r in rows])), 2), 'Sentences with ≥1 adjective': f"{np.mean([len(r['adjectives']) > 0 for r in rows]):.1%}",
         'Sources': ', '.join(f'{s} ({c})' for s, c in Counter(r.get('source', 'travel blogs / Wikipedia') for r in rows).most_common()), 'Annotation': 'authors of the base paper' if name.startswith('Base') else 'collection-time labels + independent author pass (κ in T3.5)'}
    if extra: o.update(extra)
    return o
t33 = pd.DataFrame([stats(ev106, 'Base-paper (global)'), stats(pk, 'Pakistan (local)', {'Urban / rural rows': f'{urban.sum()} / {(~urban).sum()}', 'Cities': ', '.join(f'{c} ({n})' for c, n in Counter(pk_city).most_common(6))})]).T.reset_index()
t33.columns = ['Statistic', 'Base-paper (global)', 'Pakistan (local)']; t33 = t33.iloc[1:]
save_tab(t33, '3.3', 'evalset_statistics', 'Statistics of the two evaluation sets. The frequency-prior row is the test-set-peeking oracle that always answers with the three most frequent tags; it is the bar any Hits@3 claim must clear.', 'eval/*_extracted.json', col_width=60)

# F3.3 — tag distributions, two panels, n>=3 tags in accent
f, ax = fig(7.2, 4.2, ncols=2)
for a, rows, name in zip(ax, [ev106, pk], ['Base-paper evaluation set', 'Pakistan evaluation set']):
    d = Counter(r['gold_tag'] for r in rows); items = d.most_common(); tags, cnt = zip(*items)
    cols = [C['ours'] if c >= 3 else C['base'] for c in cnt]; y = np.arange(len(tags))
    a.barh(y, cnt, color=cols, height=0.7); a.set_yticks(y); a.set_yticklabels(tags, fontsize=6 if len(tags) > 30 else 7); a.invert_yaxis(); a.grid(axis='y', visible=False)
    a.set_title(f'{name} (n={len(rows)}, {len(d)} tags)', loc='left'); a.set_xlabel('sentences')
    for yy, c in zip(y, cnt):
        if c >= 3: a.text(c, yy, f' {c}', va='center', fontsize=6.5, color=C['ink'])
ax[0].text(0.98, 0.02, 'accent = tags with ≥3 sentences\n(primary evaluation subset)', transform=ax[0].transAxes, ha='right', va='bottom', fontsize=7, color=C['muted'])
f.tight_layout(); save_fig(f, '3.3', 'tag_distribution', 'Gold-tag distribution of the two evaluation sets. Both are long-tailed; tags with at least three sentences (accent) form the primary evaluation subset on the Pakistan set.', 'eval/*_extracted.json')
print(t33.to_string(index=False, max_colwidth=60))

In [ ]:
# T3.4 — weak-label class distribution · F3.4 — map of Pakistan (corpus sentences + evaluation sentences by city) · T3.6 — train/test overlap check
import geopandas as gpd
m1 = json.load(open(f'{ROOT}/results_m/m1_summary.json'))
vc = wl.groupby(['weak_tag', 'split']).size().unstack(fill_value=0); vc['total'] = vc.sum(1); vc = vc.sort_values('total', ascending=False)
pkc = wl[wl.region == 'pk'].weak_tag.value_counts()
t34 = pd.DataFrame({'Tag': vc.index[:20], 'Train': vc['train'].values[:20], 'Dev': vc['dev'].values[:20], 'Pakistan-region rows': [int(pkc.get(t, 0)) for t in vc.index[:20]], 'Share of corpus': (vc['total'].values[:20] / vc['total'].sum())})
t34.loc[len(t34)] = ['(remaining %d tags)' % (len(vc) - 20), int(vc['train'].values[20:].sum()), int(vc['dev'].values[20:].sum()), int(sum(pkc.get(t, 0) for t in vc.index[20:])), vc['total'].values[20:].sum() / vc['total'].sum()]
t34.loc[len(t34)] = ['Total', int(vc['train'].sum()), int(vc['dev'].sum()), int(pkc.sum()), 1.0]
save_tab(t34, '3.4', 'weak_label_distribution', f'Weak-label class distribution of the training corpus ({len(vc)} OSM tags; place-based 85/15 train/dev split). Labels come from type nouns in the sentence, so the corpus is skewed toward settlements and generic venues; the Pakistan-region column counts rows geolocated inside Pakistan.', 'results_m/weak_labelled_v2.parquet, results_m/m1_summary.json', float_fmt='%.3f')

# F3.4 — map: corpus sentences (points) + evaluation sentences as count-scaled bubbles at city centroids (the evaluation rows carry no venue coordinates)
CITY = {'Karachi': (24.86, 67.01), 'Lahore': (31.55, 74.34), 'Islamabad': (33.69, 73.04), 'Rawalpindi': (33.60, 73.05), 'Peshawar': (34.01, 71.58), 'Quetta': (30.18, 66.98), 'Multan': (30.20, 71.47),
        'Faisalabad': (31.42, 73.08), 'Hyderabad': (25.39, 68.37), 'Gilgit': (35.92, 74.31), 'Hunza': (36.32, 74.65), 'Skardu': (35.29, 75.63), 'Swat': (34.77, 72.36), 'Naran': (34.91, 73.65), 'Murree': (33.90, 73.39),
        'Abbottabad': (34.15, 73.22), 'Bahawalpur': (29.40, 71.68), 'Thatta': (24.75, 67.92), 'Taxila': (33.75, 72.80), 'Sialkot': (32.49, 74.53), 'Chitral': (35.85, 71.79), 'Sukkur': (27.70, 68.86), 'Gwadar': (25.13, 62.33), 'Mardan': (34.20, 72.04), 'Kalam': (35.49, 72.58)}
pkb = gpd.read_file(f'{ROOT}/results_m/_pakistan.geojson'); corp = pd.read_parquet(f'{ROOT}/results_m/pk_from_corpus.parquet', columns=['lat', 'lon']).dropna()
cityc = Counter(str(r.get('city') or '') for r in pk); placed = {c: n for c, n in cityc.items() if c in CITY}; unplaced = {c: n for c, n in cityc.items() if c not in CITY}
f, ax = fig(5.6, 5.2); pkb.plot(ax=ax, color='#f6f6f4'); pkb.boundary.plot(ax=ax, color=C['base'], linewidth=0.8)
ax.scatter(corp.lon, corp.lat, s=3, color=C['prev'], alpha=0.45, linewidths=0, label=f'training-corpus sentences with coordinates (n={len(corp):,})')
for c, n in placed.items():
    lat, lon = CITY[c]; ax.scatter(lon, lat, s=18 + 9 * n, color=C['ours'], alpha=0.75, edgecolor='white', linewidths=0.6, zorder=3)
    if n >= 3: ax.annotate(f'{c} ({n})', (lon, lat), xytext=(5, 4), textcoords='offset points', fontsize=7, color=C['ink'], zorder=4)
ax.scatter([], [], s=40, color=C['ours'], label=f'evaluation sentences by city, area ∝ count (n={sum(placed.values())})')
ax.set_xlim(60.5, 78); ax.set_ylim(23, 37.5); ax.set_aspect('equal'); ax.grid(False); ax.set_xlabel('longitude'); ax.set_ylabel('latitude'); ax.legend(loc='lower right', fontsize=7)
ax.set_title('Pakistan: geolocated training sentences and evaluation sentences by city', loc='left')
save_fig(f, '3.4', 'pakistan_map', f'Geolocated Pakistani sentences in the training corpus (grey; {len(corp):,} of {m1["pk_from_corpus"]:,}) and the evaluation sentences aggregated by city (bubble area proportional to count). Boundary: Natural Earth admin-0.', 'results_m/pk_from_corpus.parquet, results_m/_pakistan.geojson, eval/pk_evalset_extracted.json')
print('placed', sum(placed.values()), '| not on map:', unplaced)

# T3.6 — overlap check
lk = json.load(open(f'{ROOT}/results_p/p2_leakage_check.json')); lr = json.load(open(f'{ROOT}/results_m/m5_leakage_robust.json')); m5 = json.load(open(f'{ROOT}/results_m/m5_transfer.json'))['n3_158']
rows = []
for k, v in lk.items():
    es, cs = k.split(' vs '); rows.append([es.replace('106 (base paper)', 'Base-paper (106)').replace('PK 199', 'Pakistan (199)'), cs.split(' (')[0].replace('weak_labelled_v2', 'weakly-labelled training set').replace('annotated_v2', 'full annotated corpus'), len(v['exact']), len(v['near_dup_jaccard>=0.6'])])
t36 = pd.DataFrame(rows, columns=['Evaluation set', 'Compared against', 'Exact matches', 'Near-duplicates (token Jaccard ≥ 0.6)'])
save_tab(t36, '3.6', 'overlap_check', f'Train/test overlap check. No evaluation sentence appears verbatim in either training corpus; the Pakistan set has {len(lr["excluded_rows"])} near-duplicates of weakly-labelled training sentences (rows {", ".join(map(str, lr["excluded_rows"]))}). Excluding them changes Hits@3 on the n≥3 subset from {m5["transfer v1, λ=0.35 fixed"]["Hits@3"]:.3f} to {lr["transfer_clean"]["Hits@3"]:.3f} (this work) and from {m5["W2V (Alg.2)"]["Hits@3"]:.3f} to {lr["W2V_clean"]["Hits@3"]:.3f} (previous approach).', 'results_p/p2_leakage_check.json, results_m/m5_leakage_robust.json')
print(t36.to_string(index=False))

In [ ]:
# T3.5 — annotation agreement on the Pakistan set (runs once L0.2 has ingested the author's pass)
kap = json.load(open(f'{ROOT}/results_m/m4_kappa.json'))
if 'author' not in kap.get('protocol', ''):
    print('T3.5 waiting: run L0.2 after filling eval/pk_annotation_sheet.xlsx (current m4_kappa.json is not the author pass)')
else:
    dis = kap['disagreements']; in_alt = sum(1 for d in dis if d['B'] in d['alt']); unc = sum(1 for d in dis if d['uncertain'])
    SYN = [{'cemetery', 'grave-yard'}, {'marketplace', 'market'}, {'gate', 'city-gate'}, {'monument', 'memorial'}, {'fort', 'castle'}, {'college', 'university'}, {'lake', 'pond'}, {'park', 'garden'}]
    syn = sum(1 for d in dis if any({d['A'], d['B']} <= s for s in SYN))
    n_adj = sum(1 for r in json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json')) if r.get('collection_tag', r['gold_tag']) != r['gold_tag'])
    t35 = pd.DataFrame([['Sentences', kap['n']], ["Cohen's κ (all rows)", round(kap['kappa_all'], 3)], ['Exact agreement', f"{kap['agreement_all']:.1%}"], ["Cohen's κ (n≥3 subset, %d rows)" % kap['n_n3'], round(kap['kappa_n3'], 3)],
                        ['Disagreements', len(dis)], ['… where B\'s tag is among A\'s alternatives', in_alt], ['… where A had flagged the row uncertain', unc], ['… between near-synonym tags of the lexicon', syn],
                        ['… neither in alternatives nor flagged', sum(1 for d in dis if d['B'] not in d['alt'] and not d['uncertain'])], ['Final labels changed by adjudication', n_adj]], columns=['Statistic', 'Value'])
    save_tab(t35, '3.5', 'annotation_agreement', 'Agreement between the collection-time labels (A) and the author\'s independent blind pass (B) on the Pakistan evaluation set; both passes used the closed 227-tag list and saw only the sentence text. Disagreements were resolved by the author to produce the final labels. A sizeable share of disagreements are between near-synonymous tags that the base paper\'s lexicon keeps distinct (cemetery/grave-yard, marketplace/market, gate/city-gate), which caps attainable agreement.', 'results_m/m4_kappa.json, eval/pk_evalset_extracted.json')
    ex = pd.DataFrame([[d['row'], d['sentence'][:100] + ('…' if len(d['sentence']) > 100 else ''), d['A'], d['B'], ', '.join(d['alt']) or '—'] for d in dis[:6]], columns=['Row', 'Sentence', 'Label A', 'Label B', 'A\'s alternatives'])
    save_tab(ex, '3.5b', 'annotation_disagreements', 'Examples of annotation disagreements on the Pakistan evaluation set; most are granularity or synonym choices (café vs restaurant, cemetery vs grave-yard, monument vs tower) rather than errors.', 'results_m/m4_kappa.json', col_width=52)
    print(t35.to_string(index=False))

## Stage 2 — Extraction: context nouns + adjective layer (Ch. 3)
Pipeline architecture diagram, extraction statistics on both evaluation sets, the adjective–noun pair figures from the corpus, and a worked example that follows one sentence through the pipeline.

In [ ]:
# F3.5 — pipeline architecture diagram (graphviz) · T3.7 — extraction statistics · F3.6 — corpus pair figures (re-export)
import graphviz, shutil
g = graphviz.Digraph('pipeline', format='pdf', graph_attr={'rankdir': 'LR', 'fontname': 'Helvetica', 'fontsize': '9', 'nodesep': '0.25', 'ranksep': '0.35', 'splines': 'ortho', 'pad': '0.1'},
                     node_attr={'shape': 'box', 'style': 'rounded,filled', 'fontname': 'Helvetica', 'fontsize': '9', 'fillcolor': '#f6f6f4', 'color': '#c3c2b7', 'margin': '0.12,0.06'}, edge_attr={'color': '#52514e', 'arrowsize': '0.7', 'penwidth': '0.9'})
g.node('in', 'Volunteered place description\n(informal / paraphrased text)', fillcolor='white')
with g.subgraph(name='cluster_ex') as c:
    c.attr(label='Stage 2 · Extraction (transformer parser)', style='rounded,dashed', color='#898781', fontname='Helvetica', fontsize='9')
    c.node('parse', 'Dependency parse\n(RoBERTa-based)'); c.node('nouns', 'Context nouns\n(abstract nouns removed)'); c.node('adj', 'Adjectives +\nadjective–noun pairs')
with g.subgraph(name='cluster_train') as c:
    c.attr(label='Stages 3–4 · Pattern mining & feature selection (offline, weakly-labelled corpus)', style='rounded,dashed', color='#898781', fontname='Helvetica', fontsize='9')
    c.node('corpus', 'Weakly-labelled corpus\n(470k sentences, 69 tags)', fillcolor='white'); c.node('fpg', 'Guided extended FP-Growth\n(frequent adjectives &\nadjective–noun patterns)'); c.node('ben', 'Bayesian elastic net\n(posterior weights, 94% HDI)\n— frequentist EN as comparison'); c.node('table', 'Adjective/pattern → tag\nweight table', fillcolor='#cde2fb')
with g.subgraph(name='cluster_rank') as c:
    c.attr(label='Stage 5 · Ranking', style='rounded,dashed', color='#898781', fontname='Helvetica', fontsize='9')
    c.node('sim', 'Embedding similarity\nnouns ↔ 227 OSM tags\n(Word2Vec / Numberbatch)'); c.node('blend', 'Blend\n(1−λ)·similarity + λ·pattern score', fillcolor='#cde2fb'); c.node('rank', 'Ranked OSM tags\n+ contributing patterns\n(explanation)', fillcolor='#cde2fb')
g.node('osm', 'OSM query (Overpass)\n→ candidate places → map', fillcolor='white')
g.edges([('in', 'parse'), ('parse', 'nouns'), ('parse', 'adj'), ('nouns', 'sim'), ('adj', 'blend'), ('sim', 'blend'), ('corpus', 'fpg'), ('fpg', 'ben'), ('ben', 'table'), ('table', 'blend'), ('blend', 'rank'), ('rank', 'osm')])
g.render(f'{FIG}/fig_3-5_pipeline_architecture', cleanup=True); g.format = 'png'; g.render(f'{FIG}/fig_3-5_pipeline_architecture', cleanup=True)
INDEX.append(('F3.5', 'figures/fig_3-5_pipeline_architecture.pdf', 'extraction', 'Architecture of the proposed framework: a transformer-based parser extracts context nouns and adjectives; guided extended FP-Growth and a Bayesian elastic net learn an adjective/pattern → tag weight table from a weakly-labelled corpus; at inference the table is blended with embedding similarity to rank OSM tags, each with its contributing patterns, and the top tags drive the OSM query.', 'generated (graphviz)'))
print('saved fig_3-5_pipeline_architecture')

# T3.7 — extraction statistics (recomputed from the extracted evaluation files)
stem = lambda w: re.sub(r'(ies|es|s)$', '', w.lower())
def recall_theirs(rows):
    hit = tot = 0
    for r in rows:
        theirs = {stem(w) for t in r['context_nouns'] for w in parts(t)}; ours = {stem(w) for t in r['ours_nouns'] for w in parts(t)}; hit += len(theirs & ours); tot += len(theirs)
    return hit / tot
def ext_stats(rows, name, has_theirs):
    n = len(rows); adj = [len(r['adjectives']) for r in rows]
    return {'Evaluation set': name, 'Sentences': n, 'Context nouns per sentence (base-paper extraction)': round(float(np.mean([len(r['context_nouns']) for r in rows])), 2),
            'Nouns per sentence (this work)': round(float(np.mean([len(r['ours_nouns']) for r in rows])), 2), 'Recall of the base-paper nouns (stem-matched)': f'{recall_theirs(rows):.1%}' if has_theirs else '— (nouns provided at collection)',
            'Adjectives found (total)': int(sum(adj)), 'Adjectives per sentence': round(float(np.mean(adj)), 2), 'Sentences with ≥1 adjective': f'{np.mean([a > 0 for a in adj]):.1%}',
            'Adjective–noun pairs per sentence': round(float(np.mean([sum(x.startswith('amod:') for x in r['transaction']) for r in rows])), 2), 'Sentences where a direct tag hint fires': f'{np.mean([len(r["tag_hints"]) > 0 for r in rows]):.1%}'}
t37 = pd.DataFrame([ext_stats(ev106, 'Base-paper (global)', True), ext_stats(pk, 'Pakistan (local)', False)]).T.reset_index(); t37.columns = ['Statistic', 'Base-paper (global)', 'Pakistan (local)']; t37 = t37.iloc[1:]
save_tab(t37, '3.7', 'extraction_statistics', 'Extraction statistics on the two evaluation sets. The base paper discards adjectives; 95–96% of sentences carry at least one, ~3 per sentence, which is the signal the adjective layer adds. Direct tag hints (adjective lexicon rules) fire on only 9–14% of sentences, which is why the mined weight table, not the rule list, carries the gain.', 'eval/*_extracted.json', col_width=60)
for i, (src, dst) in enumerate([('02_pair_distribution', 'fig_3-6a_pair_distribution'), ('03_top_patterns', 'fig_3-6b_top_pairs')]):
    for ext in ('pdf', 'png'): shutil.copy(f'{ROOT}/figures_blockd/{src}.{ext}', f'{FIG}/{dst}.{ext}')
    INDEX.append((f'F3.6{"ab"[i]}', f'figures/{dst}.pdf', 'extraction', ['Distribution of adjective–noun pairs per sentence in the training corpus.', 'Most frequent adjective–noun pairs in the training corpus.'][i], f'figures_blockd/{src} (generated in the annotation notebook)'))
print(t37.to_string(index=False))

In [ ]:
# F3.7 — worked example: one Pakistan sentence through the pipeline (similarity-only ranking vs blended ranking, with the adjective contributions)
rk = RK['W2V']; base = rk.ranks(pk, lam=0); full = rk.ranks(pk)
cands = [i for i in range(len(pk)) if base[i] >= 4 and full[i] == 1 and sum(a in EN_AIX for a in pk[i]['adjectives']) >= 2]
i = max(cands, key=lambda i: sum(a in EN_AIX for a in pk[i]['adjectives']) - 0.01 * len(pk[i]['sentence'])); r = pk[i]
terms = [t for t in r['context_nouns'] if t.lower() not in ABSTRACT]; s_sim = rk._norm(rk.sim(terms)); s_en = rk.en_adj(r); s_bl = (1 - LAM) * s_sim + LAM * s_en
top_sim = np.argsort(-s_sim)[:8]; top_bl = np.argsort(-s_bl)[:8]
contrib = sorted([(a, EN_CLS[int(np.argmax(COEF[:, EN_AIX[a]]))], float(COEF[:, EN_AIX[a]].max())) for a in r['adjectives'] if a in EN_AIX], key=lambda x: -x[2])
f = plt.figure(figsize=(7.4, 4.6)); gs = f.add_gridspec(2, 2, height_ratios=[1, 2.2], width_ratios=[1, 1])
a0 = f.add_subplot(gs[0, :]); a0.axis('off')
a0.text(0, 1, textwrap.fill(f'“{r["sentence"]}”', 120), va='top', fontsize=8.5, color=C['ink'])
a0.text(0, 0.42, f'context nouns: {", ".join(terms)}', va='top', fontsize=8, color=C['prev'])
a0.text(0, 0.18, f'adjectives: ' + ', '.join(f'{a} → {t} (+{w:.1f})' if w > 0 else a for a, t, w in [(a, *(next(((t, w) for aa, t, w in contrib if aa == a), ('', 0)))) for a in r['adjectives']]), va='top', fontsize=8, color=C['ours'])
a0.text(0, -0.08, f'gold tag: {r["gold_tag"]}   |   rank with nouns only: {int(base[i])}   →   rank with the adjective table (λ = {LAM}): {int(full[i])}', va='top', fontsize=8, color=C['ink'], weight='bold')
for a, idx, sc, title, col in [(f.add_subplot(gs[1, 0]), top_sim, s_sim, 'Nouns → embedding similarity only', C['prev']), (f.add_subplot(gs[1, 1]), top_bl, s_bl, f'Blended: (1−λ)·similarity + λ·adjective table', C['ours'])]:
    tags = [rk.TAGS[j] for j in idx]; y = np.arange(len(tags))
    if col == C['ours']: a.barh(y, [(1 - LAM) * s_sim[j] for j in idx], color=C['prev'], height=0.65, label='similarity part'); a.barh(y, [LAM * s_en[j] for j in idx], left=[(1 - LAM) * s_sim[j] for j in idx], color=C['ours'], height=0.65, label='adjective-table part'); a.legend(loc='lower right', fontsize=7)
    else: a.barh(y, [sc[j] for j in idx], color=col, height=0.65)
    a.set_yticks(y); a.set_yticklabels([('★ ' if t == r['gold_tag'] else '') + t for t in tags], fontsize=8); a.invert_yaxis(); a.grid(axis='y', visible=False); a.set_title(title, loc='left'); a.set_xlabel('score (min–max normalised)')
f.tight_layout(); save_fig(f, '3.7', 'worked_example', f'Worked example (Pakistan set, row {r["row"]}). With context nouns alone the gold tag “{r["gold_tag"]}” ranks {int(base[i])}; the mined adjective→tag weights ({", ".join(f"{a}→{t}" for a, t, _ in contrib[:3])}) move it to rank {int(full[i])}. ★ marks the gold tag.', 'ranker + results_i/i3_elasticnet.npz')
print(r['row'], r['sentence']); print('nouns', terms, '| adjectives', r['adjectives'], '| contributions', contrib)

## Stage 3 — Pattern mining: guided vs extended FP-Growth (Ch. 3)
Cost/output comparison of the two mining strategies, the scaling curve, and the top mined rules.

## Stage 4 — Feature selection: elastic net and Bayesian elastic net (Ch. 3)
Fit summaries, the Bayesian forest plot (credible pattern → tag weights with 94% intervals) and the inference diagnostics.

In [ ]:
# T3.8 — guided vs extended FP-Growth · F3.8 — scaling curve · T3.9 — top mined rules
i2 = json.load(open(f'{ROOT}/results_i/i2_fpgrowth_comparison.json')); j2 = json.load(open(f'{ROOT}/results_i/j2_fpgrowth_scaling.json'))
t38 = pd.DataFrame([['Extended FP-Growth (all items)', f"{i2['subsample_n']:,}", f"{i2['extended_40k']['time_s']:.1f}", f"{i2['extended_40k']['peak_mb']:.0f}", f"{i2['extended_40k']['itemsets_total']:,}", i2['extended_40k']['amod_tag_pairs']],
                    ['Guided extended FP-Growth (adjective-anchored)', f"{i2['subsample_n']:,}", f"{i2['guided_40k']['time_s']:.2f}", f"{i2['guided_40k']['peak_mb']:.0f}", '—', i2['guided_40k']['amod_tag_pairs']],
                    ['Extended FP-Growth (all items)', f"{j2[-1]['n']:,}", 'did not finish (20 min)', '—', '—', '—'],
                    ['Guided extended FP-Growth (adjective-anchored)', f"{j2[-1]['n']:,}", f"{i2['guided_full']['time_s']:.2f}", f"{i2['guided_full']['peak_mb']:.0f}", '—', i2['guided_full']['amod_tag_pairs']]],
                   columns=['Mining strategy', 'Transactions', 'Time (s)', 'Peak memory (MB)', 'Frequent itemsets', '(adjective–noun, tag) rules'])
sp = i2['extended_40k']['time_s'] / i2['guided_40k']['time_s']
save_tab(t38, '3.8', 'fpgrowth_comparison', f'Guided versus extended FP-Growth on the weakly-labelled training transactions (min. support {i2["min_support"]}). On the 40k subsample both strategies return the identical rule set; the guided variant is {sp:,.0f}× faster and needs no measurable memory, and it is the only one that completes on the full corpus. {i2["guided_rules"]["precision_lift_gt_1"]:.1%} of the {i2["guided_rules"]["n"]} rules have lift > 1.', 'results_i/i2_fpgrowth_comparison.json, results_i/j2_fpgrowth_scaling.json')
# F3.8 — scaling curve
f, ax = fig(7.0, 2.9, ncols=2)
n = [d['n'] for d in j2]; ext_t = [d['ext_time_s'] for d in j2]; gd_t = [d['guided_time_s'] for d in j2]; ext_m = [d['ext_peak_mb'] for d in j2]
ok = [d['ext_time_s'] is not None for d in j2]
ax[0].plot(np.array(n)[ok], np.array(ext_t, float)[ok], '-o', color=C['prev'], ms=4, lw=1.6, label='extended FP-Growth'); ax[0].plot(n, gd_t, '-o', color=C['ours'], ms=4, lw=1.6, label='guided extended FP-Growth')
ax[0].scatter([n[-1]], [max(ext_t[:-1]) * 3], marker='x', color=C['prev'], s=40, zorder=3); ax[0].annotate('did not finish\n(20 min budget)', (n[-1], max(ext_t[:-1]) * 3), xytext=(-8, 6), textcoords='offset points', ha='right', fontsize=7, color=C['prev'])
ax[0].set_xscale('log'); ax[0].set_yscale('log'); ax[0].set_xlabel('transactions'); ax[0].set_ylabel('mining time (s)'); ax[0].set_title('Time', loc='left'); ax[0].legend(loc='center left', fontsize=7)
ax[1].plot(np.array(n)[ok], np.array(ext_m, float)[ok], '-o', color=C['prev'], ms=4, lw=1.6, label='extended'); ax[1].plot(n, [max(d['guided_peak_mb'], 0.1) for d in j2], '-o', color=C['ours'], ms=4, lw=1.6, label='guided (≈ 0 MB)')
ax[1].set_xscale('log'); ax[1].set_yscale('log'); ax[1].set_xlabel('transactions'); ax[1].set_ylabel('peak memory (MB)'); ax[1].set_title('Memory', loc='left'); ax[1].legend(loc='center left', fontsize=7)
for a in ax: a.text(0.02, 0.02, 'identical rule sets at every size', transform=a.transAxes, fontsize=7, color=C['muted'])
f.tight_layout(); save_fig(f, '3.8', 'fpgrowth_scaling', 'Scaling of the two mining strategies with the number of training transactions (log–log). Extended FP-Growth grows in time and memory and does not complete on the full 398k transactions; guided extended FP-Growth stays below a second with negligible memory and returns the same rules.', 'results_i/j2_fpgrowth_scaling.json')
# T3.9 — top rules
rules = pd.read_parquet(f'{ROOT}/results_i/i2_guided_rules.parquet').sort_values('count', ascending=False).head(20)
t39 = pd.DataFrame({'Adjective–noun pattern': rules.pattern.str.replace('amod:', '').str.replace('_', ' '), 'Tag': rules.tag, 'Count': rules['count'].astype(int), 'Support': rules.support, 'Confidence': rules.confidence, 'Lift': rules.lift})
save_tab(t39, '3.9', 'top_mined_rules', 'The twenty most frequent adjective–noun → tag rules mined by guided extended FP-Growth from the weakly-labelled corpus, with support, confidence and lift.', 'results_i/i2_guided_rules.parquet', float_fmt='%.4f')
print(t39.head(8).to_string(index=False))

In [ ]:
# T3.10 — elastic-net fit summaries · F3.9 — Bayesian elastic net forest plot · T3.11 — Bayesian inference diagnostics
fb = json.load(open(f'{ROOT}/results_p/p1_ben_figure2.json')); fen = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True); q7 = np.load(f'{ROOT}/results_q/q7_v2_en.npz', allow_pickle=True)
def top_map(coef, feats, classes, k=6, skip_circular=True):
    sig = np.abs(coef).max(0); out = []
    for j in np.argsort(-sig):
        t = classes[int(np.argmax(coef[:, j]))]; p = feats[j].split(':', 1)[-1].replace('_', ' ')
        if skip_circular and (t in p or t.replace('-', ' ') in p): continue
        out.append(f'{p}→{t}')
        if len(out) == k: break
    return ', '.join(out)
t310 = pd.DataFrame([['Adjective-level elastic net (headline blend)', len(EN_ADJ), '150,000 (stratified)', len(EN_CLS), '—', f'{(COEF != 0).mean():.1%}', top_map(COEF, EN_ADJ, EN_CLS)],
                     ['Pattern-level elastic net (966 FP-Growth patterns)', len(PV), '120,000 (stratified)', len(PCLS), f"{float(fen['fit_time_s']):.0f}", f"{(fen['coef'] != 0).mean():.1%}", top_map(fen['coef'], PV, PCLS)],
                     ['Bayesian elastic net (same 966 patterns, NUTS)', len(PV), f"{fb['n_sub']:,} (stratified)", len(PCLS), f"{fb['time_bayes_s']:.0f}", f"{fb['pct_credible']:.1%} credible", top_map((BMU * BSEL).T, PV, PCLS)],
                     ['Adjective-level elastic net, augmented corpus (variant)', len(q7['adj_vocab']), '150,000 (stratified)', len(q7['classes']), '—', f"{(q7['coef'] != 0).mean():.1%}", top_map(q7['coef'], list(q7['adj_vocab']), list(q7['classes']))]],
                    columns=['Model', 'Features', 'Training rows', 'Tags', 'Fit time (s)', 'Non-zero / credible coefficients', 'Strongest non-circular feature → tag'])
save_tab(t310, '3.10', 'elastic_net_fits', 'Feature-selection fits. All frequentist fits use multinomial logistic regression with an elastic-net penalty (l1 ratio 0.5, C = 0.5, saga). The L1 term zeroes most coefficients; the Bayesian fit reports the share of coefficients whose 94% interval excludes zero. Circular features (the tag word itself) are skipped in the last column.', 'results_i/i3_elasticnet.npz, results_p/p1_frequentist_en.npz, results_p/p1_bayesian_en.npz, results_q/q7_v2_en.npz', col_width=44)
# F3.9 — forest plot
fig2 = fb['figure2_noncircular'][:25]
f, ax = fig(6.6, 6.0); y = np.arange(len(fig2))[::-1]
for yy, d in zip(y, fig2):
    ax.plot([d['hdi'][0], d['hdi'][1]], [yy, yy], color=C['ours'], lw=1.6, solid_capstyle='round'); ax.plot(d['mean'], yy, 'o', color=C['ours'], ms=4.5); ax.plot(d['freq_coef'], yy, 'D', color=C['prev'], ms=3.2, mfc='white', mew=1)
ax.set_yticks(y); ax.set_yticklabels([f"{d['pattern'].split(':', 1)[-1].replace('_', ' ')}  →  {d['tag']}" for d in fig2], fontsize=7.5); ax.axvline(0, color=C['base'], lw=0.8); ax.grid(axis='y', visible=False)
ax.set_xlabel('coefficient (log-odds contribution to the tag)'); ax.set_title('Bayesian elastic net: top credible pattern → tag weights', loc='left')
ax.plot([], [], 'o-', color=C['ours'], label='posterior mean with 94% interval'); ax.plot([], [], 'D', color=C['prev'], mfc='white', label='frequentist elastic-net coefficient'); ax.legend(loc='lower right', fontsize=7)
f.tight_layout(); save_fig(f, '3.9', 'bayesian_forest_plot', f'Top 25 credible, non-circular pattern → tag weights from the Bayesian elastic net ({fb["chains"]} NUTS chains × {fb["draws"]} draws on {fb["n_sub"]:,} rows): posterior mean and 94% highest-density interval, with the frequentist coefficient for comparison (Spearman ρ = {fb["spearman_rho"]:.2f} across all patterns). Every interval excludes zero; the weights read as domain knowledge.', 'results_p/p1_ben_figure2.json')
# T3.11 — diagnostics
svi = np.load(f'{ROOT}/results_p/p1_svi_en.npz', allow_pickle=True)
t311 = pd.DataFrame([['Prior', 'β = z·√τ², z ~ N(0,1), τ² ~ Exp(1/(2b²)) (⇒ Laplace(0, b)), b ~ HalfNormal(1), ridge factor exp(−λ₂ Σβ²) with λ₂ = 0.5; α ~ N(0, 2)'], ['Likelihood', 'Categorical(softmax(Xβ + α)) over 69 tags'],
                     ['Training rows (stratified subsample)', f"{fb['n_sub']:,}"], ['Patterns × tags', f'{len(PV)} × {len(PCLS)} = {len(PV) * len(PCLS):,} coefficients'], ['Sampler', 'NUTS (max tree depth 10, target accept 0.8), 2 vectorised chains × 500 warm-up + 500 draws, GPU'],
                     ['Wall time', f"{fb['time_bayes_s']:.0f} s (frequentist fit on the same patterns: {fb['time_freq_s']:.0f} s)"], ['Divergences', str(fb['divergences'])], ['R̂ over the 200 strongest patterns (median / max)', f"{fb['rhat_top200_median']:.3f} / {fb['rhat_top200_max']:.3f}"],
                     ['Effective sample size (median, same patterns)', f"{fb['ess_top200_median']:.0f}"], ['Learned Laplace scale b (posterior mean)', f"{fb['b_mean']:.3f}"], ['Coefficients with 94% interval excluding 0', f"{fb['pct_credible']:.2%} ({fb['n_selected_bayes']} of {len(PV)} patterns have ≥1)"],
                     ['Agreement with the frequentist selection', f"Jaccard {fb['jaccard']:.2f}; Spearman ρ of |coefficient| {fb['spearman_rho']:.3f}"], ['Mean-field variational approximation (for reference)', f"{int(svi['steps'])} full-batch steps in {float(svi['fit_time_s']):.0f} s; median 94% interval width {float(np.median(svi['hdi_hi'] - svi['hdi_lo'])):.2f} vs {float(np.median(BHI - BLO)):.2f} under NUTS — intervals not calibrated, so NUTS is reported"]],
                    columns=['Item', 'Value'])
save_tab(t311, '3.11', 'bayesian_diagnostics', 'Specification and convergence diagnostics of the Bayesian elastic net. The scale-mixture parameterisation is what makes the model tractable for HMC; the direct Laplace parameterisation saturates the tree depth and does not mix.', 'results_p/p1_ben_figure2.json, results_p/p1_bayesian_en.npz, results_p/p1_svi_en.npz', col_width=80)
print(t310.iloc[:, [0, 5, 6]].to_string(index=False, max_colwidth=70))

## Stages 5–6 — Ranking and evaluation protocol (Ch. 3)
Hyperparameters and constants used throughout, and the metric definitions with the two reference baselines.

In [ ]:
# T3.12 — hyperparameters & constants · T3.13 — metrics and baselines
pp = json.load(open(f'{ROOT}/results_p/p1_patterns.json')); p2c = json.load(open(f'{ROOT}/results_p/p2_classifier.json')); h3 = json.load(open(f'{ROOT}/results_h/h3_reproduction.json'))
t312 = pd.DataFrame([
    ['Extraction', 'Parser', 'spaCy en_core_web_trf (RoBERTa-base); adjectives via amod / compound / predicative links; participles keep surface form'],
    ['Extraction', 'Abstract nouns removed', ', '.join(sorted(ABSTRACT))],
    ['Pattern mining', 'Transactions', f"{pp['n_train_txn']:,} training sentences → items {{adjective, adjective–noun pair, tag}}"],
    ['Pattern mining', 'Minimum support', f"{pp['min_support']} → {len(pp['freq_adj_1itemsets'])} adjective 1-itemsets + {len(pp['freq_pair_2itemsets'])} adjective–noun 2-itemsets"],
    ['Feature selection (frequentist)', 'Model', 'multinomial logistic regression, elastic-net penalty, l1 ratio 0.5, C = 0.5, saga, max 200 iterations, tol 1e-3'],
    ['Feature selection (frequentist)', 'Adjective vocabulary', f'adjectives with ≥ 30 training occurrences → {len(EN_ADJ)} features; stratified 150,000-row subsample'],
    ['Feature selection (Bayesian)', 'Prior / sampler', 'see T3.11'],
    ['Ranking', 'Tag lexicon', f'{len(LEX)} OSM tags from the base paper (plurals collapsed)'],
    ['Ranking', 'Embeddings', 'Word2Vec GoogleNews-300 (base-paper faithful); ConceptNet Numberbatch 17.06 (best); others in the backbone ablation'],
    ['Ranking', 'Out-of-vocabulary ladder', 'surface variants (case, joined, squashed) → mean of constituent-word vectors → term dropped'],
    ['Ranking', 'Similarity score', 'per noun: cosine to all tags, min–max normalised, summed over nouns'],
    ['Ranking', 'Blend', f'score = (1 − λ)·similarity + λ·pattern score, λ = {LAM} (modal choice of 5-fold cross-validation on Hits@3; per-fold choices reported in T4.3)'],
    ['Ranking', 'Base-paper clustering (reproduction only)', f"tag–tag clustering, reading B, prune-k = 20, n = {h3['cluster_n']} nearest neighbours"],
    ['Evaluation', 'Tie rule', 'worst case: rank = number of tags scoring ≥ the gold tag'],
    ['Evaluation', 'Metrics', 'Hits@{1,2,3,5,10,20}, nDCG@20 (see T3.13); Hits@3 is the headline (k = number of OSM queries)'],
    ['Classifier ablation', 'DistilBERT', f"distilbert-base-uncased fine-tuned on {p2c['train_rows']:,} weakly-labelled rows, {p2c['epochs']} epoch, batch 64, max length 64, lr 5e-5, fp16"]],
    columns=['Stage', 'Setting', 'Value'])
save_tab(t312, '3.12', 'hyperparameters', 'Hyperparameters and constants used in the experiments.', 'code constants, results_p/p1_patterns.json, results_p/p2_classifier.json', col_width=90)
bl = h3['baselines']; mk = list(bl.keys())
t313 = pd.DataFrame([['Hits@k', 'share of sentences whose gold tag is ranked ≤ k (worst-case ties); k = number of OSM queries needed'], ['nDCG@20', 'mean of 1 / log₂(rank + 1) for rank ≤ 20, else 0 (a single relevant tag, ideal DCG = 1)'],
                     ['Majority-class baseline', f'always answers the most frequent tag of the set; base-paper set: Hits@k = {bl[mk[0]]["Hits@1"]:.3f} for all k'],
                     ['Frequency-prior baseline (oracle)', f'ranks tags by their frequency in the test set itself (test-peeking); base-paper set: Hits@1 {bl[mk[1]]["Hits@1"]:.3f}, Hits@3 {bl[mk[1]]["Hits@3"]:.3f}, Hits@5 {bl[mk[1]]["Hits@5"]:.3f}, nDCG {bl[mk[1]]["nDCG@20"]:.3f}'],
                     ['Cross-validated λ', 'blend weight chosen on four folds by Hits@3, scored on the fifth; reported numbers are the held-out concatenation']], columns=['Term', 'Definition'])
save_tab(t313, '3.13', 'metrics_and_baselines', 'Metric definitions and the two reference baselines. Both baselines are recomputed on each evaluation set; every result table carries them.', 'results_h/h3_reproduction.json', col_width=95)
print('saved T3.12, T3.13')

## Results 4.1–4.2 — Reproduction of the base paper and the main comparison (Ch. 4)
Faithfulness of the reproduction, the main results table on the base-paper evaluation set, Hits@k curves on both sets, the stage-wise ablation and the blend-weight selection.

In [ ]:
# T4.1 — reproduction · T4.2 — main table (base-paper set) · F4.1 — Hits@k curves · F4.2 — stage-wise ablation · T4.3 + F4.3 — blend-weight selection
h3 = json.load(open(f'{ROOT}/results_h/h3_reproduction.json')); h5 = json.load(open(f'{ROOT}/results_h/h5_summary.json')); j1 = json.load(open(f'{ROOT}/results_i/j1_lambda_selection.json'))
fb = json.load(open(f'{ROOT}/results_p/p1_ben_figure2.json'))['eval']; p2 = json.load(open(f'{ROOT}/results_p/p2_backbones.json'))
MK = ['Hits@1', 'Hits@2', 'Hits@3', 'Hits@5', 'Hits@10', 'Hits@20', 'nDCG@20']; PUB = h3['published']
def row(name, m): return [name] + [m.get(k, np.nan) for k in MK]
t41 = pd.DataFrame([row('Word2Vec — published', PUB['W2V']), row('Word2Vec — reproduced', h3['W2V']), row('Word2Vec + clustering — published', PUB['W2V Clustered']), row('Word2Vec + clustering — reproduced', h3['W2V_clustered'])], columns=['Model'] + MK)
d1 = np.nanmean([abs(h3['W2V'][k] - PUB['W2V'][k]) for k in PUB['W2V']]); d2 = np.nanmean([abs(h3['W2V_clustered'][k] - PUB['W2V Clustered'][k]) for k in PUB['W2V Clustered']])
save_tab(t41, '4.1', 'reproduction', f'Reproduction of the base paper on its own evaluation set (published values from Yousaf et al. 2023, Tables 4–5). Mean absolute deviation over the published cells: {d1:.3f} (Word2Vec) and {d2:.3f} (with clustering); Hits@2 and Hits@3 were not published.', 'results_h/h3_reproduction.json')
bl = h3['baselines']; bk = list(bl.keys()); w2c = h5['w2v_clustered']; nb = p2['ConceptNet Numberbatch']
t42 = pd.DataFrame([row('Majority-class baseline', bl[bk[0]]), row('Frequency-prior baseline (oracle)', bl[bk[1]]), row('Base paper, published best (Word2Vec + clustering)', PUB['W2V Clustered']),
                    row('Base paper, reproduced (Word2Vec, similarity only)', h3['W2V']), row('Base paper, reproduced (Word2Vec + clustering)', h3['W2V_clustered']),
                    row('+ abstract-noun removal', w2c['2 + abstract-noun removal']), row('+ all adjectives as extra terms (naive)', w2c['1 + all adjectives']),
                    row('+ adjective → tag table, elastic net (this work, λ by CV)', j1['cv_held_out']), row('+ pattern → tag table, elastic net (966 FP-Growth patterns)', fb['106']['freq EN (966 patterns)']['cv']),
                    row('+ pattern → tag table, Bayesian elastic net', fb['106']['Bayesian EN, HDI-selected']['cv']), row('Numberbatch embeddings + adjective → tag table (best configuration)', nb['en']['106'])], columns=['Method'] + MK)
save_tab(t42, '4.2', 'main_results_global', 'Main results on the base-paper evaluation set (106 sentences). Rows are cumulative pipeline stages on the reproduced base paper; λ is chosen by 5-fold cross-validation. Hits@3 is the headline metric.', 'results_h/*.json, results_i/j1_lambda_selection.json, results_p/p1_ben_figure2.json, results_p/p2_backbones.json')
# F4.1 — Hits@k curves (base paper vs ours) on both sets
def hits_curve(ranks): r = np.asarray(ranks, float); return [(r <= k).mean() for k in range(1, 21)]
ours106, _ = cv_ranks(RK['W2V'], ev106); nb106, _ = cv_ranks(RK['Numberbatch'], ev106)
basepk = RK['W2V'].ranks(pk, lam=0)[SUB]; ourspk = cv_ranks(RK['W2V'], pk)[0][SUB]; nbpk = cv_ranks(RK['Numberbatch'], pk)[0][SUB]
d106 = Counter(x['gold_tag'] for x in ev106); fp106 = np.array([sum(c >= d106[r['gold_tag']] for c in d106.values()) for r in ev106], float); fp = np.array([sum(c >= dpk[r['gold_tag']] for c in dpk.values()) for r in pk], float)[SUB]
f, ax = fig(7.2, 3.1, ncols=2); ks = range(1, 21)
for a, series, fpc, title in [(ax[0], [('base paper, similarity only (reproduced)', h3['ranks_plain'], C['prev'], '--'), ('base paper, + clustering (reproduced)', h3.get('ranks_clustered', h3['ranks_plain']), C['prev'], '-'), ('this work: Word2Vec + adjective table', ours106, C['ours'], '-'), ('this work: Numberbatch + adjective table', nb106, C['ours2'], '-')], fp106, 'Base-paper evaluation set (n=106)'),
                              (ax[1], [('base paper, similarity only (reproduced)', basepk, C['prev'], '-'), ('this work: Word2Vec + adjective table', ourspk, C['ours'], '-'), ('this work: Numberbatch + adjective table', nbpk, C['ours2'], '-')], fp, f'Pakistan evaluation set (n≥3 subset, n={int(SUB.sum())})')]:
    for name, rk_, col, ls in series: a.plot(ks, hits_curve(rk_), ls, color=col, lw=1.8, label=name)
    a.plot(ks, hits_curve(fpc), ':', color=C['muted'], lw=1.2, label='frequency-prior baseline (oracle)'); a.axvline(3, color=C['base'], lw=0.8); a.text(3.2, 0.03, 'k = 3 (headline)', fontsize=7, color=C['muted'])
    a.set_xlabel('k (number of OSM queries)'); a.set_ylabel('Hits@k'); a.set_ylim(0, 1); a.set_xlim(1, 20); a.set_title(title, loc='left'); a.legend(loc='lower right', fontsize=6.5)
f.tight_layout(); save_fig(f, '4.1', 'hits_at_k_curves', 'Hits@k for k = 1…20 on both evaluation sets: reproduced base paper (grey), this work with the base-paper-faithful Word2Vec embeddings (blue) and with Numberbatch (orange), and the frequency-prior oracle (dotted). The vertical line marks the headline k = 3.', 'results_h/h3_reproduction.json + ranker (λ by 5-fold CV)')
# F4.2 — stage-wise ablation bars (Hits@3 and nDCG)
arms = t42.iloc[[4, 5, 6, 7, 9, 10]]; labels = ['base paper\n(reproduced)', '+ abstract-noun\nremoval', '+ all adjectives\n(naive)', '+ adjective→tag\ntable (EN)', '+ pattern→tag\ntable (Bayesian EN)', 'Numberbatch +\nadjective table']
f, ax = fig(7.2, 3.0, ncols=2)
for a, m, fpv in [(ax[0], 'Hits@3', bl[bk[1]]['Hits@3']), (ax[1], 'nDCG@20', bl[bk[1]]['nDCG@20'])]:
    v = arms[m].values.astype(float); cols = [C['prev'], C['prev'], C['naive'], C['ours'], C['ours'], C['ours2']]; x = np.arange(len(v)); a.bar(x, v, color=cols, width=0.62)
    for xi, vi in zip(x, v): a.text(xi, vi + 0.006, f'{vi:.3f}', ha='center', fontsize=7, color=C['ink'])
    a.axhline(fpv, color=C['muted'], ls=':', lw=1.1); a.text(len(v) - 0.5, fpv + 0.006, 'frequency-prior oracle', ha='right', fontsize=6.5, color=C['muted'])
    pub = PUB['W2V Clustered'].get(m)
    if pub is not None: a.axhline(pub, color=C['prev'], ls='--', lw=0.9); a.text(-0.4, pub - 0.03, 'published best', fontsize=6.5, color=C['prev'])
    a.set_xticks(x); a.set_xticklabels(labels, fontsize=6.5); a.set_ylabel(m); a.set_title(m + ', base-paper evaluation set', loc='left'); a.grid(axis='x', visible=False); a.set_ylim(0, max(v.max(), fpv) + 0.08)
f.tight_layout(); save_fig(f, '4.2', 'stagewise_ablation', 'Effect of each pipeline stage on the base-paper evaluation set (Hits@3 left, nDCG@20 right). Adding adjectives as plain extra terms (green) hurts; the mined adjective → tag table (blue) is what produces the gain, and Numberbatch embeddings (orange) add a little more. Dashed: published best (nDCG only; Hits@3 was not published); dotted: frequency-prior oracle.', 'T4.2')
# T4.3 + F4.3 — λ selection
curve = {float(k): v for k, v in j1['full_curve'].items()}; lams = sorted(curve)
t43 = pd.DataFrame([[L, curve[L]['Hits@1'], curve[L]['Hits@3'], curve[L]['Hits@5'], curve[L]['nDCG@20'], j1['folds_chose'].count(L)] for L in lams], columns=['λ', 'Hits@1', 'Hits@3', 'Hits@5', 'nDCG@20', 'Folds choosing λ'])
save_tab(t43, '4.3', 'lambda_selection', f'Blend weight λ: full-set metrics at each candidate value and how many of the five cross-validation folds selected it (folds chose {j1["folds_chose"]}). The held-out concatenation gives Hits@3 {j1["cv_held_out"]["Hits@3"]:.3f}; the modal λ = {LAM} is used everywhere else.', 'results_i/j1_lambda_selection.json')
f, ax = fig(4.6, 2.8); ax.plot(lams, [curve[L]['Hits@3'] for L in lams], '-o', color=C['ours'], ms=4, lw=1.8, label='Hits@3'); ax.plot(lams, [curve[L]['nDCG@20'] for L in lams], '-s', color=C['ours2'], ms=4, lw=1.8, label='nDCG@20')
for L in set(j1['folds_chose']): ax.axvline(L, color=C['base'], lw=0.8, ls='--')
ax.text(LAM + 0.005, 0.44, f'CV picks\n(λ = {", ".join(map(str, sorted(set(j1["folds_chose"]))))})', fontsize=7, color=C['muted']); ax.set_xlabel('λ (weight of the adjective → tag table)'); ax.set_ylabel('score'); ax.legend(loc='lower right', fontsize=7); ax.set_title('Blend weight on the base-paper set', loc='left')
f.tight_layout(); save_fig(f, '4.3', 'lambda_curve', 'Hits@3 and nDCG@20 on the base-paper set as a function of the blend weight λ (λ = 0 is the base paper). Dashed lines mark the values selected by the cross-validation folds; the curve is flat between 0.3 and 0.4, so the choice is not a cherry-pick.', 'results_i/j1_lambda_selection.json')
print(t42[['Method', 'Hits@1', 'Hits@3', 'nDCG@20']].to_string(index=False))

## Results 4.3–4.4 — Ablations and the Bayesian selection (Ch. 4)
Embedding-backbone ablation, the fine-tuned classifier arm, pattern granularity, training-corpus variants, and the Bayesian-vs-frequentist comparison.

In [ ]:
# T4.4 + F4.4 — backbone ablation · T4.5 — classifier arm · T4.6 — pattern granularity · T4.7 — corpus variants · T4.8 — Bayesian vs frequentist
p2 = json.load(open(f'{ROOT}/results_p/p2_backbones.json')); p2c = json.load(open(f'{ROOT}/results_p/p2_classifier.json')); p2g = json.load(open(f'{ROOT}/results_p/p2_granularity.json')); q6 = json.load(open(f'{ROOT}/results_q/q6_v1_vs_v3.json')); q7 = json.load(open(f'{ROOT}/results_q/q7_v1_vs_v2.json'))
m5v2 = json.load(open(f'{ROOT}/results_m/m5_transfer_v2.json')); m5 = json.load(open(f'{ROOT}/results_m/m5_transfer.json')); fbf = json.load(open(f'{ROOT}/results_p/p1_ben_figure2.json')); j1 = json.load(open(f'{ROOT}/results_i/j1_lambda_selection.json'))
COND = {'nouns': 'nouns only', 'naive': '+ all adjectives (naive)', 'en': '+ adjective → tag table'}; BB = [b for b in p2 if isinstance(p2[b], dict)]; p2g = {k: v for k, v in p2g.items() if isinstance(v, dict)}
rows = []
for b in BB:
    for c in COND: rows.append([b, COND[c]] + [p2[b][c][s][k] for s in ('106', 'PK n>=3') for k in ('Hits@1', 'Hits@3', 'nDCG@20')])
t44 = pd.DataFrame(rows, columns=['Embedding backbone', 'Condition', 'Global Hits@1', 'Global Hits@3', 'Global nDCG@20', 'Pakistan Hits@1', 'Pakistan Hits@3', 'Pakistan nDCG@20'])
wins = sum(p2[b]['en'][s]['Hits@3'] > p2[b]['nouns'][s]['Hits@3'] for b in BB for s in ('106', 'PK n>=3')); wins_n = sum(p2[b]['en'][s]['nDCG@20'] > p2[b]['nouns'][s]['nDCG@20'] for b in BB for s in ('106', 'PK n>=3')); hurts = sum(p2[b]['naive'][s]['Hits@3'] < p2[b]['nouns'][s]['Hits@3'] for b in BB for s in ('106', 'PK n>=3'))
save_tab(t44, '4.4', 'backbone_ablation', f'Embedding-backbone ablation: seven frozen embedding models × three conditions on both evaluation sets (Pakistan: n≥3 subset; λ by 5-fold CV). The adjective → tag table improves nDCG on {wins_n}/14 and Hits@3 on {wins}/14 backbone–set pairs; naive adjectives lower Hits@3 on {hurts}/14. Sentence encoders are the weakest backbones for single-term similarity.', 'results_p/p2_backbones.json')
f, ax = fig(7.4, 3.4, ncols=2); x = np.arange(len(BB)); w = 0.26
for a, s, title in [(ax[0], '106', 'Base-paper evaluation set'), (ax[1], 'PK n>=3', 'Pakistan evaluation set (n≥3 subset)')]:
    for i, (c, col) in enumerate([('nouns', C['prev']), ('naive', C['naive']), ('en', C['ours'])]):
        v = [p2[b][c][s]['Hits@3'] for b in BB]; a.bar(x + (i - 1) * w, v, w, color=col, label=COND[c])
        if c == 'en':
            for xi, vi in zip(x, v): a.text(xi + w, vi + 0.008, f'{vi:.2f}', ha='center', fontsize=6, color=C['ink'])
    a.set_xticks(x); a.set_xticklabels([b.replace(' (', '\n(').replace('ConceptNet ', '') for b in BB], fontsize=6.5); a.set_ylabel('Hits@3'); a.set_title(title, loc='left'); a.grid(axis='x', visible=False); a.set_ylim(0, 0.72)
ax[0].legend(loc='upper left', fontsize=6.5)
f.tight_layout(); save_fig(f, '4.4', 'backbone_ablation', 'Hits@3 per embedding backbone under the three conditions. On every backbone and both sets, naive adjectives (green) fall below nouns only (grey) while the mined adjective → tag table (blue) rises above it; Numberbatch + table is the best configuration.', 'results_p/p2_backbones.json')
# T4.5 — classifier arm
t45 = pd.DataFrame([['Weak-label dev split (69 tags, 5,000 rows)', f"top-1 accuracy {p2c['dev_acc_weak']:.3f}", '—', '—', '—'],
                    ['Base-paper set — classifier only', f"{p2c['106']['classifier_only']['Hits@1']:.3f}", f"{p2c['106']['classifier_only']['Hits@3']:.3f}", f"{p2c['106']['classifier_only']['Hits@5']:.3f}", f"{p2c['106']['classifier_only']['nDCG@20']:.3f}"],
                    ['Base-paper set — classifier + adjective → tag table (fusion, λ by CV)', f"{p2c['106']['classifier_plus_en_cv']['Hits@1']:.3f}", f"{p2c['106']['classifier_plus_en_cv']['Hits@3']:.3f}", f"{p2c['106']['classifier_plus_en_cv']['Hits@5']:.3f}", f"{p2c['106']['classifier_plus_en_cv']['nDCG@20']:.3f}"],
                    ['Base-paper set — similarity ranker + adjective → tag table (this work)', f"{j1['cv_held_out']['Hits@1']:.3f}", f"{j1['cv_held_out']['Hits@3']:.3f}", f"{j1['cv_held_out']['Hits@5']:.3f}", f"{j1['cv_held_out']['nDCG@20']:.3f}"],
                    ['Pakistan set (n≥3) — classifier only', f"{p2c['PK n>=3']['classifier_only']['Hits@1']:.3f}", f"{p2c['PK n>=3']['classifier_only']['Hits@3']:.3f}", f"{p2c['PK n>=3']['classifier_only']['Hits@5']:.3f}", f"{p2c['PK n>=3']['classifier_only']['nDCG@20']:.3f}"],
                    ['Pakistan set (n≥3) — classifier + adjective → tag table (fusion)', f"{p2c['PK n>=3']['classifier_plus_en_cv']['Hits@1']:.3f}", f"{p2c['PK n>=3']['classifier_plus_en_cv']['Hits@3']:.3f}", f"{p2c['PK n>=3']['classifier_plus_en_cv']['Hits@5']:.3f}", f"{p2c['PK n>=3']['classifier_plus_en_cv']['nDCG@20']:.3f}"],
                    ['Pakistan set (n≥3) — similarity ranker + adjective → tag table (this work)', f"{m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@1']:.3f}", f"{m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@3']:.3f}", f"{m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@5']:.3f}", f"{m5['n3_158']['transfer v1, λ=0.35 fixed']['nDCG@20']:.3f}"]],
                   columns=['Arm', 'Hits@1', 'Hits@3', 'Hits@5', 'nDCG@20'])
save_tab(t45, '4.5', 'classifier_arm', f'Fine-tuned DistilBERT classifier ({p2c["train_rows"]:,} weakly-labelled sentences, {p2c["epochs"]} epoch). It reaches {p2c["dev_acc_weak"]:.1%} on held-out weak labels — which name the type — but collapses on paraphrased descriptions; the gold tag lies inside its 69 classes for {p2c["106"]["gold_coverage_69"]:.0%} (global) and {p2c["PK n>=3"]["gold_coverage_69"]:.0%} (Pakistan) of sentences, so coverage is not the cause.', 'results_p/p2_classifier.json, results_i/j1_lambda_selection.json, results_m/m5_transfer.json', col_width=70)
# T4.6 — granularity
t46 = pd.DataFrame([[k.replace(' (P1.3)', ''), f"{v['fit_s']:.0f}", f"{v['pct_nonzero']:.0%}", f"{v['106']['coverage']:.0%}", v['106']['Hits@1'], v['106']['Hits@3'], v['106']['nDCG@20'], f"{v['PK n>=3']['coverage']:.0%}", v['PK n>=3']['Hits@1'], v['PK n>=3']['Hits@3'], v['PK n>=3']['nDCG@20']] for k, v in p2g.items()],
                   columns=['Pattern family', 'Fit (s)', 'Non-zero', 'Global coverage', 'Global Hits@1', 'Global Hits@3', 'Global nDCG', 'Pakistan coverage', 'Pakistan Hits@1', 'Pakistan Hits@3', 'Pakistan nDCG'])
save_tab(t46, '4.6', 'pattern_granularity', 'Pattern granularity: the elastic net refit on adjective 1-itemsets only, adjective–noun 2-itemsets only, or both (same 120,000-row subsample; coverage = share of evaluation sentences with at least one active pattern). Adjectives carry the gain; pairs alone rarely fire on paraphrased text but add to adjectives on the Pakistan set.', 'results_p/p2_granularity.json')
# T4.7 — corpus variants
t47 = pd.DataFrame([['Wikivoyage only (398k rows) — the pipeline', q7['v1_cv']['Hits@1'], q7['v1_cv']['Hits@3'], q7['v1_cv']['nDCG@20'], m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@1'], m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@3'], m5['n3_158']['transfer v1, λ=0.35 fixed']['nDCG@20']],
                    [f"+ filtered Wikidata ({q7['aug_rows']:,} adjective-bearing rows, {len(q7['aug_tags'])} under-represented tags)", q7['v2_cv']['Hits@1'], q7['v2_cv']['Hits@3'], q7['v2_cv']['nDCG@20'], m5v2['n3_158']['transfer v2, λ=0.35']['Hits@1'], m5v2['n3_158']['transfer v2, λ=0.35']['Hits@3'], m5v2['n3_158']['transfer v2, λ=0.35']['nDCG@20']],
                    ['+ all Wikidata (497k rows; nouns only)', q6['v3_baseline']['Hits@1'], q6['v3_baseline']['Hits@3'], q6['v3_baseline']['nDCG@20'], np.nan, np.nan, np.nan],
                    ['+ all Wikidata (497k rows; + pattern → tag table)', q6['v3_frequentist_blend']['Hits@1'], q6['v3_frequentist_blend']['Hits@3'], q6['v3_frequentist_blend']['nDCG@20'], np.nan, np.nan, np.nan]],
                   columns=['Training corpus', 'Global Hits@1', 'Global Hits@3', 'Global nDCG@20', 'Pakistan Hits@1', 'Pakistan Hits@3', 'Pakistan nDCG@20'])
dv2 = round((m5v2['n3_158']['transfer v2, λ=0.35']['Hits@3'] - m5['n3_158']['transfer v1, λ=0.35 fixed']['Hits@3']) * m5.get('n_n3', 157))
save_tab(t47, '4.7', 'corpus_variants', f'Training-corpus variants. Adding all Wikidata descriptions (encyclopaedic register) lowers every metric; adding only adjective-bearing Wikidata rows for under-represented tags gains three sentences on the global set but changes the Pakistan result by {dv2:+d} sentences. Register match with the target text matters more than corpus size or label cleanliness, so the Wikivoyage-only corpus is kept.', 'results_q/q6_v1_vs_v3.json, results_q/q7_v1_vs_v2.json, results_m/m5_transfer_v2.json')
# T4.8 — Bayesian vs frequentist
E = fbf['eval']
t48 = pd.DataFrame([['Patterns with ≥1 non-zero / credible weight', f"{fbf['n_selected_freq']} of {len(PV)}", f"{fbf['n_selected_bayes']} of {len(PV)}"], ['Coefficients selected', 'non-zero (L1)', f"{fbf['pct_credible']:.2%} with 94% interval excluding 0"], ['Fit time', f"{fbf['time_freq_s']:.0f} s (CPU)", f"{fbf['time_bayes_s']:.0f} s (GPU, 2 chains)"],
                    ['Agreement of selections', '—', f"Jaccard {fbf['jaccard']:.2f}; Spearman ρ {fbf['spearman_rho']:.3f}"], ['Uncertainty per weight', 'none', '94% highest-density interval'],
                    ['Global set: Hits@1 / Hits@3 / nDCG (λ by CV)', ' / '.join(f"{E['106']['freq EN (966 patterns)']['cv'][k]:.3f}" for k in ('Hits@1', 'Hits@3', 'nDCG@20')), ' / '.join(f"{E['106']['Bayesian EN, HDI-selected']['cv'][k]:.3f}" for k in ('Hits@1', 'Hits@3', 'nDCG@20'))],
                    ['Pakistan set (n≥3): Hits@1 / Hits@3 / nDCG (λ by CV)', ' / '.join(f"{E['PK n>=3']['freq EN (966 patterns)']['cv'][k]:.3f}" for k in ('Hits@1', 'Hits@3', 'nDCG@20')), ' / '.join(f"{E['PK n>=3']['Bayesian EN, HDI-selected']['cv'][k]:.3f}" for k in ('Hits@1', 'Hits@3', 'nDCG@20'))]],
                   columns=['', 'Frequentist elastic net', 'Bayesian elastic net'])
save_tab(t48, '4.8', 'bayesian_vs_frequentist', 'Bayesian versus frequentist elastic net on the same 966 FP-Growth patterns. The two select largely the same patterns and rank equally well within noise; the Bayesian fit adds a calibrated interval to every pattern → tag weight, which is what the explanations in Section 4.7 report.', 'results_p/p1_ben_figure2.json')
print(t44[t44.Condition == '+ adjective → tag table'][['Embedding backbone', 'Global Hits@3', 'Pakistan Hits@3']].to_string(index=False))

## Results 4.5 — Regional transfer: the Pakistan evaluation set (Ch. 4)
Baselines, the base-paper method, the transferred pipeline and its locally-trained variants; the urban/rural slice; per-tag gains.

In [ ]:
# T4.9 + F4.5 — Pakistan results · T4.9b — urban vs rural · F4.6 — per-tag Hits@3
m5r = json.load(open(f'{ROOT}/results_m/m5_region_experiments.json'))
NAMES = [('majority class', 'Majority-class baseline'), ('freq prior (oracle)', 'Frequency-prior baseline (oracle)'), ('W2V (Alg.2)', 'Base paper (Word2Vec, similarity only)'), ('W2V-Clustered (Alg.3)', 'Base paper (Word2Vec + clustering)'),
         ('transfer v1, λ=0.35 fixed', 'This work, transferred unchanged (global training, λ = 0.35)'), ('local-only, λ by CV', 'This work, trained on Pakistan rows only (2,235)'), ('merged global+PK, λ by CV', 'This work, global + Pakistan rows'), ('merged, PK x10, λ by CV', 'This work, global + Pakistan rows up-weighted ×10')]
def r2(name, m): return [name] + [m[k] for k in MK]
t49 = pd.DataFrame([r2(lbl + f' — {"n≥3 subset" if s == "n3_158" else "all 199"}', m5r[s][k]) for s in ('n3_158', 'full_199') for k, lbl in NAMES] + [r2('Numberbatch + adjective → tag table — n≥3 subset', nb['en']['PK n>=3'])], columns=['Method'] + MK)
save_tab(t49, '4.9', 'pakistan_results', 'Results on the Pakistan evaluation set (primary: 23 tags with ≥3 sentences, 158 rows; secondary: all 199 rows). The base paper\'s clustering step hurts on this 54-tag set. The pipeline transferred without any Pakistani training data already beats the base paper on every metric; local variants land within a few sentences of it.', 'results_m/m5_region_experiments.json, results_p/p2_backbones.json')
f, ax = fig(7.2, 3.0); keys = [k for k, _ in NAMES][2:] + ['numberbatch']; lab = ['base paper\n(similarity)', 'base paper\n(+ clustering)', 'transferred\nunchanged', 'Pakistan rows\nonly', 'global +\nPakistan', 'global +\nPakistan ×10', 'Numberbatch +\ntable']
v3 = [m5r['n3_158'][k]['Hits@3'] for k in keys[:-1]] + [nb['en']['PK n>=3']['Hits@3']]; vn = [m5r['n3_158'][k]['nDCG@20'] for k in keys[:-1]] + [nb['en']['PK n>=3']['nDCG@20']]
x = np.arange(len(lab)); cols = [C['prev'], C['prev'], C['ours'], C['ours'], C['ours'], C['ours'], C['ours2']]
ax.bar(x - 0.19, v3, 0.36, color=cols, label='Hits@3'); ax.bar(x + 0.19, vn, 0.36, color=cols, alpha=0.45, label='nDCG@20')
for xi, a3, an in zip(x, v3, vn): ax.text(xi - 0.19, a3 + 0.008, f'{a3:.2f}', ha='center', fontsize=6.5); ax.text(xi + 0.19, an + 0.008, f'{an:.2f}', ha='center', fontsize=6.5, color=C['muted'])
fpv = m5r['n3_158']['freq prior (oracle)']['Hits@3']; ax.axhline(fpv, color=C['muted'], ls=':', lw=1.1); ax.text(len(lab) - 0.5, fpv + 0.01, 'frequency-prior oracle (Hits@3)', ha='right', fontsize=6.5, color=C['muted'])
ax.set_xticks(x); ax.set_xticklabels(lab, fontsize=6.5); ax.set_ylim(0, 0.75); ax.grid(axis='x', visible=False); ax.set_title('Pakistan evaluation set, n≥3 subset', loc='left'); ax.legend(loc='upper left', fontsize=7)
f.tight_layout(); save_fig(f, '4.5', 'pakistan_results', 'Hits@3 (solid) and nDCG@20 (light) on the Pakistan n≥3 subset. Every variant of this work (blue/orange) beats both base-paper arms (grey); training locale changes the result by only a few sentences, and the frequency-prior oracle is far below.', 'results_m/m5_region_experiments.json, results_p/p2_backbones.json')
# T4.9b — urban vs rural slices (base vs ours, W2V; same λ=0.35 transfer arm)
urb = np.array([str(r.get('city') or '') in URBAN for r in pk]); base_all = RK['W2V'].ranks(pk, lam=0); ours_all = RK['W2V'].ranks(pk); nb_all = RK['Numberbatch'].ranks(pk)
rows = []
for sl, mask in [('Urban (major cities)', urb), ('Rural / small towns / valleys', ~urb)]:
    for nm, rk_ in [('Base paper (similarity only)', base_all), ('This work (Word2Vec + table)', ours_all), ('This work (Numberbatch + table)', nb_all)]:
        m = metrics(rk_[mask]); rows.append([sl, int(mask.sum()), nm, m['Hits@1'], m['Hits@3'], m['Hits@5'], m['nDCG@20']])
t49b = pd.DataFrame(rows, columns=['Slice', 'Sentences', 'Method', 'Hits@1', 'Hits@3', 'Hits@5', 'nDCG@20'])
save_tab(t49b, '4.9b', 'pakistan_urban_rural', f'Urban versus rural slices of the Pakistan set (all 199 rows; urban = {urb.sum()} sentences from the major cities, rural = {(~urb).sum()} from valleys, small towns and the northern areas). The rural slice is small, so its numbers are indicative only.', 'eval/pk_evalset_extracted.json + ranker')
# F4.6 — per-tag Hits@3 on the n>=3 tags, base vs ours, sorted by gain
tags3 = [t for t, c in dpk.most_common() if c >= 3]; g = np.array([r['gold_tag'] for r in pk])
per = pd.DataFrame([[t, int((g == t).sum()), (base_all[g == t] <= 3).mean(), (ours_all[g == t] <= 3).mean()] for t in tags3], columns=['tag', 'n', 'base', 'ours']); per['gain'] = per.ours - per.base; per = per.sort_values('gain', ascending=False)
f, ax = fig(6.4, 4.6); y = np.arange(len(per))
ax.hlines(y, per.base, per.ours, color=C['base'], lw=2); ax.plot(per.base, y, 'o', color=C['prev'], ms=5, label='base paper (similarity only)'); ax.plot(per.ours, y, 'o', color=C['ours'], ms=5, label='this work (Word2Vec + adjective table)')
ax.set_yticks(y); ax.set_yticklabels([f'{t} (n={n})' for t, n in zip(per.tag, per.n)], fontsize=7.5); ax.invert_yaxis(); ax.set_xlabel('Hits@3 within the tag'); ax.set_xlim(-0.02, 1.05); ax.grid(axis='y', visible=False); ax.legend(loc='lower right', fontsize=7); ax.set_title('Per-tag Hits@3 on the Pakistan set (tags with ≥3 sentences)', loc='left')
f.tight_layout(); save_fig(f, '4.6', 'pakistan_per_tag', f'Per-tag Hits@3 on the Pakistan set for the 23 tags with at least three sentences, sorted by gain. The adjective table helps most on culturally-marked types ({", ".join(per.tag.head(4))}) and never removes a tag from the top three entirely.', 'eval/pk_evalset_extracted.json + ranker')
print(t49b.to_string(index=False)); print(per.head(6).to_string(index=False))

## Results 4.6 — Error analysis (Ch. 4)
Confusion matrix on the base-paper set and the sentences the adjective stage fixes or breaks, on both sets, with the adjective → tag weight responsible.

## Results 4.8 — Multilingual input: Urdu and Roman-Urdu (Ch. 4)
The same pipeline behind a translation front-end. Every variant is scored through the identical fully automatic path (transformer extractor → nouns/adjectives → ranker), so the difference between rows is the translation and nothing else.

In [ ]:
# T4.15 + F4.8 — translation loss · T4.16 — the held-bucket audit
n2 = json.load(open(f'{ROOT}/results_n/n2_translation_loss.json')); n3 = json.load(open(f'{ROOT}/results_n/n3_held_urdu.json')); V = n2['variants']
LBL = {'English original (collection nouns)': 'English original (hand-picked context nouns)', 'English original (extractor nouns)': 'English original (automatic extraction)',
       'Urdu → NLLB-200 → English': 'Urdu (hand-written) → NLLB-200 → English', 'Urdu → LLM → English': 'Urdu (hand-written) → instruction-tuned LLM → English', 'Roman-Urdu → LLM → English': 'Roman-Urdu (hand-written) → instruction-tuned LLM → English'}
t415 = pd.DataFrame([[LBL[k], n2['n_roman'] if k.startswith('Roman') else n2['n_rows']] + [V[k][bb][c][m] for bb in ('W2V', 'Numberbatch') for c in ('nouns_only', 'with_table') for m in (['Hits@3'] if c == 'nouns_only' else ['Hits@1', 'Hits@3', 'nDCG@20'])] for k in V],
                    columns=['Input', 'Sentences', 'W2V nouns-only Hits@3', 'W2V Hits@1', 'W2V Hits@3', 'W2V nDCG@20', 'NB nouns-only Hits@3', 'NB Hits@1', 'NB Hits@3', 'NB nDCG@20'])
auto, urdu_n = V['English original (extractor nouns)']['W2V']['with_table']['Hits@3'], V['Urdu → NLLB-200 → English']['W2V']['with_table']['Hits@3']
ru, ru_en = V['Roman-Urdu → LLM → English']['W2V']['with_table']['Hits@3'], n2['english_on_roman_rows']['W2V']['Hits@3']
save_tab(t415, '4.15', 'translation_loss', f'Multilingual input. The author hand-wrote Urdu for all {n2["n_rows"]} sentences of the primary Pakistan subset and Roman-Urdu for {n2["n_roman"]} of them; each variant is then machine-translated to English and pushed through the unchanged pipeline. Against the fully automatic English ceiling ({auto:.3f}), Urdu costs {auto - urdu_n:.3f} Hits@3; on the {n2["n_roman"]} rows that have both, Roman-Urdu reaches {ru:.3f} against {ru_en:.3f} for the English originals. The adjective → tag table improves every translated variant, so the contribution survives the language front-end.', 'results_n/n2_translation_loss.json')
f, ax = fig(7.0, 3.1); ks = list(V); x = np.arange(len(ks)); w = 0.38
for i, (c, col, lab) in enumerate([('nouns_only', C['prev'], 'nouns only (base-paper stage)'), ('with_table', C['ours'], '+ adjective → tag table')]):
    v = [V[k]['W2V'][c]['Hits@3'] for k in ks]; ax.bar(x + (i - 0.5) * w, v, w, color=col, label=lab)
    for xi, vi in zip(x, v): ax.text(xi + (i - 0.5) * w, vi + 0.008, f'{vi:.2f}', ha='center', fontsize=6.5, color=C['ink'])
ax.axhline(auto, color=C['muted'], ls=':', lw=1.1); ax.text(len(ks) - 0.5, auto + 0.01, 'automatic-English ceiling', ha='right', fontsize=6.5, color=C['muted'])
ax.set_xticks(x); ax.set_xticklabels([LBL[k].replace(' → ', '\n→ ').replace(' (', '\n(') for k in ks], fontsize=6.3); ax.set_ylabel('Hits@3'); ax.set_ylim(0, 0.72); ax.grid(axis='x', visible=False)
ax.legend(loc='upper right', fontsize=7); ax.set_title('Pakistan evaluation set (primary subset): cost of the translation front-end', loc='left')
f.tight_layout(); save_fig(f, '4.8', 'translation_loss', f'Hits@3 on the primary Pakistan subset for each input language. Hand-written Urdu and Roman-Urdu are machine-translated and then handled by the unchanged pipeline; the dotted line is the same pipeline on the English originals with automatic extraction. The mined adjective → tag table (blue) lifts every variant above the nouns-only stage (grey), including the translated ones.', 'results_n/n2_translation_loss.json')
A = n3['audit']
t416 = pd.DataFrame([['Rows held out by the language router', A['bucket_rows']], ['… genuine Urdu script', A['urdu_script']], ['… Latin script', A['latin_script']],
                     ['… Latin-script rows with ≥2 Roman-Urdu marker words', A['latin_with_2plus_roman_urdu_markers']], ['Urdu-script rows put through the front-end', len(n3['urdu_rows'])]], columns=['', 'Sentences'])
save_tab(t416, '4.16', 'held_bucket_audit', 'Audit of the sentences the language router held out of the English corpus. They were initially recorded as Urdu and Roman-Urdu; inspection shows only a handful are Urdu script, the remainder being English text from non-English-speaking destinations that the router could not classify because of foreign proper nouns. The corpus therefore provides no usable Urdu training material, which is why the multilingual evaluation rests on hand-written translations.', 'results_n/n3_held_urdu.json')
ex = pd.DataFrame([[o['urdu'][:60], o['english'][:80], ', '.join(o['nouns'][:5]), ', '.join(o['top3'])] for o in n3['urdu_rows'][:8]], columns=['Urdu (corpus)', 'Machine translation', 'Extracted nouns', 'Top-3 tags'])
save_tab(ex, '4.16b', 'held_urdu_examples', 'The Urdu-script corpus sentences put through the translation front-end. Well-formed sentences are typed sensibly (village → town/village, college → university/college, motorway → motorway); fragments and photo captions translate poorly and produce unusable rankings.', 'results_n/n3_held_urdu.json', col_width=46)
print(t415[['Input', 'Sentences', 'W2V nouns-only Hits@3', 'W2V Hits@3', 'NB Hits@3']].to_string(index=False))

In [ ]:
# F4.7 — confusion matrix (base-paper set, tags with ≥2 sentences) · T4.10 / T4.11 — sentences fixed or broken by the adjective stage, global and Pakistan
rk = RK['W2V']
def top1(r, lam): s = rk.scores(r, lam=lam); return rk.TAGS[int(np.argmax(s))]
def why(r, tag):   # adjective -> weight toward the given tag (positive contributions, strongest first)
    if tag not in EN_CLS: return '—'
    k = EN_CLS.index(tag); c = [(a, float(COEF[k, EN_AIX[a]])) for a in r['adjectives'] if a in EN_AIX and COEF[k, EN_AIX[a]] > 0]
    return ', '.join(f'{a}→{tag} (+{w:.1f})' for a, w in sorted(c, key=lambda x: -x[1])[:3]) or '(no positive adjective weight; similarity part moved)'
def fixed_broken(rows):
    out = []
    for r in rows:
        b, o = top1(r, 0), top1(r, LAM)
        if o == r['gold_tag'] and b != r['gold_tag']: out.append(['fixed', r['row'], r['sentence'][:95] + ('…' if len(r['sentence']) > 95 else ''), r['gold_tag'], b, why(r, r['gold_tag'])])
        elif b == r['gold_tag'] and o != r['gold_tag']: out.append(['broken', r['row'], r['sentence'][:95] + ('…' if len(r['sentence']) > 95 else ''), r['gold_tag'], o, why(r, o)])
    return pd.DataFrame(out, columns=['Effect', 'Row', 'Sentence', 'Gold tag', 'Top-1 without the table (fixed) / with it (broken)', 'Adjective weights toward the winning tag'])
gold = [r['gold_tag'] for r in ev106]; pred = [top1(r, LAM) for r in ev106]; predb = [top1(r, 0) for r in ev106]
seen = [t for t, c in Counter(gold).most_common() if c >= 2]; cm = pd.DataFrame(0, index=seen, columns=seen + ['other'])
for p, g_ in zip(pred, gold):
    if g_ in seen: cm.loc[g_, p if p in seen else 'other'] += 1
f, ax = fig(5.8, 5.0); M = cm.values / cm.values.sum(1, keepdims=True)
im = ax.imshow(M, cmap=matplotlib.colors.LinearSegmentedColormap.from_list('seq', ['#ffffff'] + SEQ), vmin=0, vmax=1)
ax.set_xticks(range(len(cm.columns))); ax.set_xticklabels(cm.columns, rotation=60, ha='right', fontsize=7); ax.set_yticks(range(len(cm.index))); ax.set_yticklabels(cm.index, fontsize=7); ax.grid(False)
for i in range(len(cm.index)):
    for j in range(len(cm.columns)):
        if cm.values[i, j]: ax.text(j, i, int(cm.values[i, j]), ha='center', va='center', fontsize=6.5, color='white' if M[i, j] > 0.55 else C['ink'])
ax.set_xlabel('predicted (top-1)'); ax.set_ylabel('gold'); ax.set_title(f'Top-1 confusion, base-paper set (tags with ≥2 sentences); accuracy {np.mean([p == g_ for p, g_ in zip(pred, gold)]):.3f} vs {np.mean([p == g_ for p, g_ in zip(predb, gold)]):.3f} without the table', loc='left', fontsize=8)
cb = f.colorbar(im, ax=ax, fraction=0.035, pad=0.02); cb.set_label('row share', fontsize=7); cb.ax.tick_params(labelsize=6)
f.tight_layout(); save_fig(f, '4.7', 'confusion_matrix', 'Top-1 confusion matrix of the full pipeline on the base-paper set (rows = gold, columns = predicted, tags with at least two sentences; counts shown, shading = row share). Remaining confusions are mostly within the food-venue group (restaurant / café / pub / fast-food), which the base paper\'s own annotation notes flag as ambiguous.', 'eval/evalset_106_extracted.json + ranker')
t410 = fixed_broken(ev106); save_tab(t410, '4.10', 'fixed_broken_global', f'Base-paper set: sentences whose top-1 prediction the adjective → tag table fixes ({(t410.Effect == "fixed").sum()}) or breaks ({(t410.Effect == "broken").sum()}), with the adjective weights toward the winning tag.', 'ranker + results_i/i3_elasticnet.npz', col_width=52)
t411 = fixed_broken(pk); save_tab(t411, '4.11', 'fixed_broken_pakistan', f'Pakistan set: sentences whose top-1 prediction the adjective → tag table fixes ({(t411.Effect == "fixed").sum()}) or breaks ({(t411.Effect == "broken").sum()}), with the adjective weights toward the winning tag.', 'ranker + results_i/i3_elasticnet.npz', col_width=52)
print(f'global fixed {(t410.Effect == "fixed").sum()} / broken {(t410.Effect == "broken").sum()} | Pakistan fixed {(t411.Effect == "fixed").sum()} / broken {(t411.Effect == "broken").sum()}'); print(t411[t411.Effect == 'fixed'].head(8).iloc[:, [1, 3, 4, 5]].to_string(index=False))

## Results 4.9 — Statistical significance of the main comparisons (Ch. 4)
Three comparisons, fixed in advance (so no multiple-comparison correction is owed): the base-paper reproduction against the adjective → tag table on the base-paper set; the base-paper method against the transferred pipeline on the Pakistan set; and the frequentist against the Bayesian elastic net. Each is a *paired* comparison — both arms score the identical sentences — so the test uses the per-sentence outcomes: a paired bootstrap for the confidence interval on the difference, and McNemar's exact test on the Hits@3 indicators.

In [ ]:
# T4.17 — significance of the pre-specified comparisons (paired bootstrap CI + McNemar exact test)
from scipy.stats import binomtest
from itertools import groupby
B, ALPHA = 10000, 0.05; rng = np.random.default_rng(2026)
ev106 = json.load(open(f'{ROOT}/eval/evalset_106_extracted.json')); pk = json.load(open(f'{ROOT}/eval/pk_evalset_extracted.json'))
dpk = Counter(r['gold_tag'] for r in pk); SUB = np.array([dpk[r['gold_tag']] >= 3 for r in pk]); W = RK['W2V']
h3 = json.load(open(f'{ROOT}/results_h/h3_reproduction.json')); PC = {c: k for k, c in enumerate(PCLS)}
def ndcg_vec(r): r = np.asarray(r, float); return np.where(r <= 20, 1 / np.log2(r + 1), 0.0)
def paired(name, set_name, ra, rb, label_a, label_b):
    """ra, rb = per-sentence ranks of the two arms over the SAME sentences (paired design)."""
    n = len(ra); ha, hb = (np.asarray(ra) <= 3).astype(int), (np.asarray(rb) <= 3).astype(int); na, nb = ndcg_vec(ra), ndcg_vec(rb)
    idx = rng.integers(0, n, size=(B, n))                       # one resample of sentence indices, applied to both arms
    dh = hb[idx].mean(1) - ha[idx].mean(1); dn = nb[idx].mean(1) - na[idx].mean(1)
    lo_h, hi_h = np.percentile(dh, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)]); lo_n, hi_n = np.percentile(dn, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
    b_only, a_only = int(((hb == 1) & (ha == 0)).sum()), int(((ha == 1) & (hb == 0)).sum())   # discordant pairs only
    p = binomtest(b_only, b_only + a_only, 0.5).pvalue if (b_only + a_only) else 1.0
    return {'comparison': name, 'set': set_name, 'n': n, 'arm_a': label_a, 'arm_b': label_b,
            'hits3_a': float(ha.mean()), 'hits3_b': float(hb.mean()), 'delta_hits3': float(hb.mean() - ha.mean()), 'ci_hits3': [float(lo_h), float(hi_h)],
            'ndcg_a': float(na.mean()), 'ndcg_b': float(nb.mean()), 'delta_ndcg': float(nb.mean() - na.mean()), 'ci_ndcg': [float(lo_n), float(hi_n)],
            'mcnemar_fixed': b_only, 'mcnemar_broken': a_only, 'p_value': float(p), 'significant_at_0.05': bool(p < ALPHA and lo_h * hi_h > 0), 'bootstrap_resamples': B}
def ranks_clustered(rk, rows):            # base paper Algorithm 3 (reading B, n=4, prune 20) — the locked constants
    out = []
    for r in rows:
        s = rk._norm(rk.sim([t for t in r['context_nouns'] if t.lower() not in ABSTRACT])); ranked = [rk.TAGS[i] for i in np.argsort(-s, kind='stable')]
        pr = ranked[:20]; ps = set(pr); S = rk.M[[rk.IX[t] for t in pr]] @ rk.M.T
        for i, t in enumerate(pr): S[i, rk.IX[t]] = -np.inf
        votes = Counter({t: 0 for t in pr})
        for i in range(len(pr)):
            for j in np.argsort(-S[i])[:4]:
                if rk.TAGS[j] in ps: votes[rk.TAGS[j]] += 1
        g = r['gold_tag']; seen = 0; rank = len(rk.TAGS) + 1
        for _, grp in groupby(sorted(votes.items(), key=lambda kv: -kv[1]), key=lambda kv: kv[1]):
            cl = [t for t, _ in grp]; seen += len(cl)
            if g in cl: rank = seen; break
        out.append(rank)
    return np.array(out, float)
def pen(coefPC):                          # pattern-level table as a blend term
    c2w = {c: W.IX[c] for c in PCLS if c in W.IX}
    def en(r):
        s = np.zeros(len(W.TAGS), np.float32); f = pat_feats(r)
        if not f: return s
        agg = coefPC[f].sum(0)
        for c, w in c2w.items(): s[w] = agg[PC[c]]
        return W._norm(s)
    return en
def ranks_pen(rows, en):
    out = []
    for r in rows:
        s = W._norm(W.sim([t for t in r['context_nouns'] if t.lower() not in ABSTRACT])); s = (1 - LAM) * s + LAM * en(r)
        g = r['gold_tag']; out.append(int((s >= s[W.IX[g]]).sum()) if g in W.IX else len(W.TAGS) + 1)
    return np.array(out, float)
FC = np.load(f'{ROOT}/results_p/p1_frequentist_en.npz', allow_pickle=True)['coef'].T
pk_plain, pk_ours, pk_clu = W.ranks(pk, lam=0)[SUB], W.ranks(pk)[SUB], ranks_clustered(W, pk)[SUB]
RES = [
    paired('adjective → tag table vs base-paper reproduction (plain)', 'Base-paper set (106)', np.asarray(h3['ranks_plain'], float), cv_ranks(W, ev106)[0], 'base paper, similarity only', 'this work (adjective → tag table, λ by CV)'),
    paired('adjective → tag table vs base-paper BEST (+ clustering)', 'Base-paper set (106)', np.asarray(h3['ranks_clustered'], float), cv_ranks(W, ev106)[0], 'base paper, best arm (+ clustering)', 'this work (adjective → tag table, λ by CV)'),
    paired('adjective → tag table vs base-paper reproduction (plain)', 'Pakistan set (n≥3)', pk_plain, pk_ours, 'base paper, similarity only', 'this work (adjective → tag table, λ=0.35)'),
    paired('adjective → tag table vs base-paper BEST (+ clustering)', 'Pakistan set (n≥3)', pk_clu, pk_ours, 'base paper, best arm (+ clustering)', 'this work (adjective → tag table, λ=0.35)'),
    paired('base-paper clustering step vs plain similarity', 'Pakistan set (n≥3)', pk_plain, pk_clu, 'base paper, similarity only', 'base paper, + clustering'),
    paired('Bayesian vs frequentist elastic net (same 966 patterns)', 'Base-paper set (106)', ranks_pen(ev106, pen(FC)), ranks_pen(ev106, pen(BMU * BSEL)), 'frequentist elastic net', 'Bayesian elastic net (HDI-selected)')]
json.dump({'alpha': ALPHA, 'bootstrap_resamples': B, 'method': 'paired bootstrap over sentence indices (identical resample applied to both arms) + McNemar exact test on the paired Hits@3 indicators', 'comparisons': RES},
          open(f'{ROOT}/results_p/significance.json', 'w'), indent=1)
t417 = pd.DataFrame([[c['comparison'], c['set'], c['n'], f"{c['hits3_a']:.3f}", f"{c['hits3_b']:.3f}", f"{c['delta_hits3']:+.3f}", f"[{c['ci_hits3'][0]:+.3f}, {c['ci_hits3'][1]:+.3f}]",
                      f"{c['mcnemar_fixed']} / {c['mcnemar_broken']}", f"{c['p_value']:.4f}", 'yes' if c['significant_at_0.05'] else 'no'] for c in RES],
                    columns=['Comparison', 'Evaluation set', 'n', 'Hits@3 before', 'Hits@3 after', 'Δ Hits@3', '95% CI (paired bootstrap)', 'Sentences fixed / broken', 'McNemar p', 'Significant (α=.05)'])
save_tab(t417, '4.17', 'significance', f'Significance of the pre-specified comparisons. The interval is a paired bootstrap over sentence indices ({B:,} resamples, the same resample applied to both arms); the p-value is McNemar\'s exact test on the paired Hits@3 indicators, which uses only the sentences on which the two arms disagree. The improvement over the base paper is significant on the larger, independently collected Pakistan set against both of its arms, and against its plain variant on the base-paper set; against its clustered variant on the 106-sentence set the direction is positive but the difference lies within sampling error. The clustering step itself does not transfer — it is significantly harmful on the Pakistan set. The Bayesian and frequentist elastic nets are statistically indistinguishable, so the Bayesian fit is justified by its calibrated intervals rather than by accuracy.', 'results_p/significance.json', col_width=34)
for c in RES: print(f"{c['comparison'][:52]:<52} {c['set']:<22} Δ {c['delta_hits3']:+.3f}  CI [{c['ci_hits3'][0]:+.3f},{c['ci_hits3'][1]:+.3f}]  fixed/broken {c['mcnemar_fixed']:>2}/{c['mcnemar_broken']:<2} p={c['p_value']:.4f}  -> {'SIGNIFICANT' if c['significant_at_0.05'] else 'not significant'}")

## Results 4.7 — Explanations for end-users (Ch. 4, RQ3)
What the framework shows a user with each prediction: the ranked tags and, for each, the mined patterns that contributed, with their posterior weight and 94% interval. No post-hoc explanation method is involved — these are the model's own coefficients.

In [ ]:
# T4.12 — explanation examples · F4.10 — explanation panel · T4.13 — explanation coverage  (Bayesian pattern → tag table, W2V ranker, λ = 0.35)
rk = RK['W2V']; PC = {c: k for k, c in enumerate(PCLS)}
def explain(r, topn=3, per_tag=3):
    s = rk.scores(r, en='ben'); order = np.argsort(-s)[:topn]; f = pat_feats(r); out = []
    for j in order:
        t = rk.TAGS[j]; contrib = []
        if t in PC:
            k = PC[t]; contrib = sorted([(PV[p], float(BMU[p, k]), float(BLO[p, k]), float(BHI[p, k])) for p in f if BSEL[p, k] and BMU[p, k] > 0], key=lambda x: -x[1])[:per_tag]
        out.append((t, float(s[j]), contrib))
    return out
def fmt_pat(p): return p.split(':', 1)[-1].replace('_', ' ')
def fmt_contrib(c): return '; '.join(f'{fmt_pat(p)} +{m:.1f} [{lo:.1f}, {hi:.1f}]' for p, m, lo, hi in c) or '(similarity of the nouns only)'
rng = np.random.default_rng(11)
pick = [r for r in ev106 if explain(r)[0][2]]; pick = [pick[i] for i in rng.choice(len(pick), 4, replace=False)]
pickpk = [r for r in pk if explain(r)[0][2]]; pickpk = [pickpk[i] for i in rng.choice(len(pickpk), 4, replace=False)]
rows = []
for setname, rr in [('Base-paper', pick), ('Pakistan', pickpk)]:
    for r in rr:
        ex = explain(r); rows.append([setname, r['row'], r['sentence'][:90] + ('…' if len(r['sentence']) > 90 else ''), r['gold_tag']] + [f'{i + 1}. {t}: {fmt_contrib(c)}' for i, (t, sc, c) in enumerate(ex)])
t412 = pd.DataFrame(rows, columns=['Set', 'Row', 'Sentence', 'Gold tag', 'Rank 1 — contributing patterns [94% interval]', 'Rank 2', 'Rank 3'])
save_tab(t412, '4.12', 'explanation_examples', 'Explanations as shown to an end-user: the three top-ranked tags for a sentence, each with the mined adjective / adjective–noun patterns that contributed, their posterior weight and 94% interval. Where no pattern is credible for a tag, the tag was ranked by noun similarity alone and the explanation says so.', 'ranker + results_p/p1_bayesian_en.npz', col_width=56)
# F4.10 — explanation panel for one Pakistan sentence (correctly ranked, most contributing patterns)
cands = [r for r in pk if explain(r)[0][0] == r['gold_tag'] and len(explain(r)[0][2]) >= 2]
r = max(cands, key=lambda r: len(explain(r)[0][2]) - 0.002 * len(r['sentence'])); ex = explain(r, topn=5, per_tag=4)
f = plt.figure(figsize=(7.2, 4.4)); gs = f.add_gridspec(1, 2, width_ratios=[1, 1.35])
a = f.add_subplot(gs[0]); tags = [t for t, _, _ in ex]; sc = [s for _, s, _ in ex]; y = np.arange(len(tags)); a.barh(y, sc, color=[C['ours'] if t == r['gold_tag'] else C['base'] for t in tags], height=0.6)
a.set_yticks(y); a.set_yticklabels([('★ ' if t == r['gold_tag'] else '') + t for t in tags]); a.invert_yaxis(); a.grid(axis='y', visible=False); a.set_xlabel('blended score'); a.set_title('Ranked OSM tags', loc='left')
b = f.add_subplot(gs[1]); b.axis('off'); b.set_title('Why: contributing patterns (posterior weight, 94% interval)', loc='left')
txt = textwrap.fill(f'“{r["sentence"]}”', 70) + '\n\n'
for i, (t, s_, c) in enumerate(ex[:3]):
    txt += f'{i + 1}. {t}\n' + (''.join(f'     {fmt_pat(p):<24} +{m:.2f}   [{lo:.2f}, {hi:.2f}]\n' for p, m, lo, hi in c) if c else '     (noun similarity only)\n') + '\n'
b.text(0, 1, txt, va='top', fontsize=7.6, family='monospace', color=C['ink'], transform=b.transAxes)
f.tight_layout(); save_fig(f, '4.10', 'explanation_panel', f'Explanation panel for one Pakistan sentence (row {r["row"]}, gold “{r["gold_tag"]}”): the ranked tags (left) and, for the top three, the patterns that contributed with their posterior weight and 94% interval (right). This is the view the web application renders; nothing is computed after the fact.', 'ranker + results_p/p1_bayesian_en.npz')
# T4.13 — coverage
def cov_stats(rows, name):
    E = [explain(r, topn=1) for r in rows]; has = [len(e[0][2]) > 0 for e in E]; n = [len(e[0][2]) for e in E]
    gold_has = [r['gold_tag'] in PC and any(BSEL[p, PC[r['gold_tag']]] and BMU[p, PC[r['gold_tag']]] > 0 for p in pat_feats(r)) for r in rows]
    return [name, len(rows), f'{np.mean(has):.1%}', f'{np.mean(n):.2f}', f'{np.mean(gold_has):.1%}', f'{np.mean([len(pat_feats(r)) > 0 for r in rows]):.1%}']
t413 = pd.DataFrame([cov_stats(ev106, 'Base-paper'), cov_stats(pk, 'Pakistan')], columns=['Set', 'Sentences', 'Top-1 tag has ≥1 credible contributing pattern', 'Mean contributing patterns (top-1)', 'Gold tag has ≥1 credible contributing pattern', 'Sentences with ≥1 mined pattern'])
save_tab(t413, '4.13', 'explanation_coverage', 'How often an explanation is available: share of sentences whose top-ranked tag (and whose gold tag) has at least one credible contributing pattern, and the mean number of such patterns. Sentences without any mined pattern are ranked by noun similarity alone and are labelled as such.', 'ranker + results_p/p1_bayesian_en.npz')
print(t413.to_string(index=False)); print(r['row'], r['sentence'][:90])
for t, _, c in ex[:3]: print(' ', t, fmt_contrib(c))

In [ ]:
# Code map — which notebook and cell produces each figure, table and result file → writeup/CODE_MAP.md
import json, re, os, glob
NB = '/content/drive/MyDrive/Colab Notebooks'
STAGE = {'ab': '1. Collection (Wikivoyage / Commons / OSM)', 'cd': '2. Cleaning, annotation, extraction', 'efg': '0. Superseded prototype (geocell formulation)',
         'h': '3. Base-paper reproduction + adjective ablation', 'i': '4. Pattern mining + elastic net + final ablation', 'q': '5. Multi-source corpus experiments',
         'm (1)': '6. Pakistan corpus filtering', 'p': '6a. Bayesian elastic net (first attempts)', 'm': '7. Pakistan track, corpus variant, Bayesian EN, backbone ablation',
         'n': '8. Multilingual front-end', 'l': '9. Figures, tables, significance'}
rows, produced = [], {}
for p in sorted(glob.glob(f'{NB}/sprint_b*.ipynb')):
    key = re.sub(r'^sprint_b(?:l?o|o)ck_|\.ipynb$', '', os.path.basename(p)); nb = json.load(open(p)); name = os.path.basename(p)
    for i, c in enumerate(nb['cells']):
        if c['cell_type'] != 'code': continue
        src = ''.join(c['source'])
        if not src.strip(): continue
        label = (re.match(r'#\s*([^\n]{0,90})', src.strip()).group(1).strip() if src.strip().startswith('#') else src.strip().splitlines()[0][:60])
        ids = [('T' if kind == 'tab' else 'F') + iid for kind, iid in re.findall(r"save_(tab|fig)\([^,]+,\s*'([^']+)'", src)]
        ids += re.findall(r"INDEX\.append\(\(\s*'([FT][\d.]+[a-z]?)'", src)          # re-exported figures and the graphviz diagram
        outs = sorted({o for o in re.findall(r"(?:results_[a-z]+|eval|annotated|models_cache|figures|tables|raw_q|clean|writeup)/[A-Za-z0-9_.\-]+\.(?:json|npz|parquet|pkl|csv|md|py)", src)
                       if any(w in src for w in ('open(', 'dump', 'to_parquet', 'savez', 'save_npz', 'write'))})
        if not ids and not outs: continue
        rows.append({'notebook': name, 'stage': STAGE.get(key, key), 'cell': i, 'label': label, 'ids': sorted(set(ids)), 'writes': outs})
        for x in set(ids): produced.setdefault(x, []).append(f'`{name}` cell {i}')
idx_ids = [l.split('|')[1].strip() for l in open(f'{FIG}/INDEX.md').read().splitlines() if l.startswith('| ') and l.split('|')[1].strip() not in ('id', '---')]
md = ['# Code map — where every artifact comes from', '',
      'Generated by scanning the notebooks on Drive. Regenerate with the last cell of the figures notebook (`sprint_block_l.ipynb`).',
      'Notebook file names are historical — they record the order the work was done in. The stage column is what each one actually does.', '',
      '## Notebooks, in pipeline order', '', '| stage | notebook | what it does |', '|---|---|---|']
seen = set()
for r in sorted(rows, key=lambda r: r['stage']):
    if r['notebook'] in seen: continue
    seen.add(r['notebook']); md.append(f'| {r["stage"].split(".")[0]} | `{r["notebook"]}` | {r["stage"].split(". ", 1)[-1]} |')
md += ['', '## Figure / table → notebook and cell', '',
       'Open the notebook, jump to the cell number. Every one of these reads a file from `results/` — no number is typed by hand.', '',
       '| id | produced by |', '|---|---|'] + [f'| {i} | {", ".join(produced.get(i, ["— not found"]))} |' for i in idx_ids]
md += ['', '## Every cell that writes an artifact', '']
cur = None
for r in sorted(rows, key=lambda r: (r['stage'], r['cell'])):
    if r['stage'] != cur: cur = r['stage']; md += ['', f'### {cur}  ·  `{r["notebook"]}`', '', '| cell | what it does | produces |', '|---|---|---|']
    md.append(f'| {r["cell"]} | {r["label"]} | {", ".join(r["ids"])}{" · " if r["ids"] and r["writes"] else ""}{", ".join("`" + w + "`" for w in r["writes"][:4])} |')
os.makedirs(f'{ROOT}/writeup', exist_ok=True); open(f'{ROOT}/writeup/CODE_MAP.md', 'w').write('\n'.join(md) + '\n')
missing = [i for i in idx_ids if i not in produced]
print(f'wrote writeup/CODE_MAP.md — {len(rows)} artifact-producing cells across {len(seen)} notebooks')
print('ids with no producing cell found:', missing or 'none')

## Index and verification
Writes `figures/INDEX.md` (id, file, stage, draft caption, source) and checks that every planned figure/table exists and that no internal run labels leak into the index.

In [ ]:
# Write the index and verify completeness
import re, os
INDEX[:] = list({e[0]: e for e in INDEX}.values())
write_index()                                    # merges with whatever is already in INDEX.md, so a partial re-run is safe
PLANNED = ['F3.1', 'F3.2', 'F3.3', 'F3.4', 'F3.5', 'F3.6a', 'F3.6b', 'F3.7', 'F3.8', 'F3.9', 'T3.1', 'T3.2', 'T3.3', 'T3.4', 'T3.5', 'T3.5b', 'T3.6', 'T3.7', 'T3.8', 'T3.9', 'T3.10', 'T3.11', 'T3.12', 'T3.13',
           'F4.1', 'F4.2', 'F4.3', 'F4.4', 'F4.5', 'F4.6', 'F4.7', 'F4.8', 'F4.10', 'T4.1', 'T4.2', 'T4.3', 'T4.4', 'T4.5', 'T4.6', 'T4.7', 'T4.8', 'T4.9', 'T4.9b', 'T4.10', 'T4.11', 'T4.12', 'T4.13', 'T4.15', 'T4.16', 'T4.16b', 'T4.17']
have = {i for i, *_ in INDEX}; missing = [p for p in PLANNED if p not in have]; extra = sorted(have - set(PLANNED))
print(f'{len(have)} items indexed | planned {len(PLANNED)} | missing: {missing or "none"} | extra: {extra or "none"}   (F4.9 was dropped with the web app)')
bad = [(i, f) for i, f, *_ in INDEX if not os.path.exists(f'{ROOT}/{f}')]; print('files missing on Drive:', bad or 'none')
idx = open(f'{FIG}/INDEX.md').read(); leaks = re.findall(r'(?<![\w/])(Block [A-Z]|[HIJKMPQ]\d\.\d|Sprint)', '\n'.join(l.split('|')[4] for l in idx.splitlines() if l.startswith('| ')))
print('internal run labels in captions:', leaks or 'none')
print(f'figures: {len([f for f in os.listdir(FIG) if f.endswith(".pdf")])} PDF | tables: {len([f for f in os.listdir(TAB) if f.endswith(".tex")])} TeX / {len([f for f in os.listdir(TAB) if f.endswith(".csv")])} CSV / {len([f for f in os.listdir(TAB) if f.endswith(".png")])} PNG')
orphan = sorted(set(f.rsplit('.', 1)[0] for f in os.listdir(TAB)) - {f.split("/")[-1].rsplit('.', 1)[0] for _, f, *_ in INDEX if f.startswith('tables/')}) + sorted(set(f.rsplit('.', 1)[0] for f in os.listdir(FIG) if f.endswith('.pdf')) - {f.split("/")[-1].rsplit('.', 1)[0] for _, f, *_ in INDEX if f.startswith('figures/')})
print('files on Drive not in the index (stale, safe to delete):', orphan or 'none')